# Validate the analytics views

## Purpose

Check the analytics views before any number reaches the dashboard. The six official business-question views:

| View | Question |
| --- | --- |
| `analytics_regional_investment` | Q1. Regional investment distribution |
| `analytics_category_portfolio` | Q2. Infrastructure category portfolio |
| `analytics_project_delivery_status` | Q3. Project delivery and status |
| `analytics_population_equity` | Q4. Population-based equity |
| `analytics_flood_control_exposure` | Q5. Flood control versus flood exposure |
| `analytics_region_category` | Q6. Investment by area and type |

The five exploratory monitoring views (D-54, proposed):

| View | One row per |
| --- | --- |
| `analytics_infrastructure_year_trend` | `infra_year`, including `NULL` |
| `analytics_infrastructure_year_sector` | `infra_year`, sector, and mapping status |
| `analytics_delivery_cohort` | `infra_year`, status group, and status mapping status |
| `analytics_project_size_concentration` | Official region |
| `analytics_implementing_office_portfolio` | `implementing_office`, including `NULL` |

Each view is rebuilt here straight from Gold and compared row by row. The notebook only reads Gold and the analytics views. It never changes them. Evidence is saved to `04-validation.analytics_dq_results` **before** the STOP gate runs, so a failed check stays visible. The table has the same columns as `gold_dq_results`, with `layer = 'analytics'`.

## Inputs

- `03-gold` facts and dimensions, to recompute every view on its own
- The eleven analytics views in `05-analytics`
- `04-validation.gold_dq_results`, the Gold STOP gate

## Cost

Each Gold table is read a few times at most. The project fact is the largest. All checks come from one metrics row per view, and the gate reads the saved results.

## 1. Set the expected values and read the runs

**What:** set the expected values once, then read the Gold runs behind the analytics views.

**Why:** the rule version is `analytics-validation-v2` since the exploratory views and their checks were added (D-54, proposed). The run-ID recipe is unchanged (D-52), so v1 evidence stays as it was. The expected values live here, not in the analytics notebook, so the views never check themselves. The validation run ID comes from the Gold runs, so a rerun on the same Gold data updates the same evidence instead of adding copies.

**Expected result:** one row, with `project_snapshot_count` 1 and `project_snapshot_null_rows` 0.

In [ ]:
%sql
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `04-validation`;

DECLARE OR REPLACE VARIABLE validation_rule_version STRING DEFAULT 'analytics-validation-v2';
-- the approved status groups (d-40) and the source status d-46 asks a zero-progress count to disclose
DECLARE OR REPLACE VARIABLE finished_status_group STRING DEFAULT 'FINISHED';
DECLARE OR REPLACE VARIABLE active_status_group STRING DEFAULT 'ACTIVE';
DECLARE OR REPLACE VARIABLE inactive_status_group STRING DEFAULT 'INACTIVE';
DECLARE OR REPLACE VARIABLE for_procurement_source_status STRING DEFAULT 'For Procurement';
-- the coverage statuses the views may publish. only reported regions get a rank or a ratio
DECLARE OR REPLACE VARIABLE reported_coverage_status STRING DEFAULT 'REPORTED';
DECLARE OR REPLACE VARIABLE limited_coverage_status STRING DEFAULT 'LIMITED_SOURCE_COVERAGE';
DECLARE OR REPLACE VARIABLE no_project_coverage_status STRING DEFAULT 'NO_MAPPED_PROJECT_DATA';
-- gold exposure statuses (silver_region_flood_exposure.md) plus the view label for a region with no exposure row
DECLARE OR REPLACE VARIABLE additive_exposure_status STRING DEFAULT 'ADDITIVE_WITHIN_TOLERANCE';
DECLARE OR REPLACE VARIABLE expected_exposure_statuses ARRAY<STRING> DEFAULT ARRAY(
    'ADDITIVE_WITHIN_TOLERANCE', 'NON_ADDITIVE_CROSS_LEVEL_OVERLAP', 'NOT_EVALUATED', 'NO_EXPOSURE_DATA'
);
-- mgb levels for the project-point groups in q5
DECLARE OR REPLACE VARIABLE high_flood_levels ARRAY<STRING> DEFAULT ARRAY('High', 'Very High');
DECLARE OR REPLACE VARIABLE low_flood_levels ARRAY<STRING> DEFAULT ARRAY('Low', 'Moderate');
-- rounding slack for a share total. 18 shares rounded to 4 decimals stay well inside it
DECLARE OR REPLACE VARIABLE share_total_tolerance_pct DOUBLE DEFAULT 0.01;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_context AS
WITH project_fact AS (
    SELECT
        MAX(run_id) AS project_fact_run_id,
        MIN(source_snapshot_id) AS project_snapshot_id,
        -- the gate below needs exactly one snapshot and no empty id
        COUNT(DISTINCT source_snapshot_id) AS project_snapshot_count,
        COUNT_IF(source_snapshot_id IS NULL) AS project_snapshot_null_rows
    FROM `03-gold`.fact_project_snapshot
    WHERE is_current_snapshot
),

region_dimension AS (
    SELECT MAX(run_id) AS dim_region_run_id
    FROM `03-gold`.dim_region
),

population_fact AS (
    SELECT MAX(run_id) AS population_fact_run_id
    FROM `03-gold`.fact_region_population
),

exposure_fact AS (
    -- the newest exposure run, the same rule the q5 view uses
    SELECT
        run_id AS exposure_fact_run_id,
        mgb_source_snapshot_id
    FROM `03-gold`.fact_region_flood_exposure
    QUALIFY ROW_NUMBER() OVER (ORDER BY source_load_ts DESC, run_id) = 1
)

SELECT
    project_fact.*,
    region_dimension.dim_region_run_id,
    population_fact.population_fact_run_id,
    exposure_fact.exposure_fact_run_id,
    exposure_fact.mgb_source_snapshot_id,
    -- the same gold runs always give the same id, so a rerun updates its evidence
    SHA2(CONCAT_WS(
        '|',
        project_fact.project_fact_run_id,
        region_dimension.dim_region_run_id,
        population_fact.population_fact_run_id,
        exposure_fact.exposure_fact_run_id,
        validation_rule_version
    ), 256) AS validation_run_id
FROM project_fact
CROSS JOIN region_dimension
CROSS JOIN population_fact
LEFT JOIN exposure_fact
    ON TRUE;

SELECT * FROM analytics_validation_context;

### One current DPWH snapshot

**What:** stop unless current Gold holds exactly one DPWH snapshot, with no empty snapshot ID.

**Why:** the context reads one snapshot ID, and every check compares the views with it. If current Gold mixed two snapshots, both the views and this notebook would add them together and the checks could still pass. This gate runs before any check uses the context, so no evidence is saved for a mixed snapshot. The analytics notebook runs the same gate before it builds the views.

**Expected result:** the `ASSERT_TRUE` returns `NULL`, and no error is raised.

In [ ]:
%sql
SELECT ASSERT_TRUE(
    project_snapshot_count = 1 AND project_snapshot_null_rows = 0,
    'STOP: current Gold project rows span multiple DPWH snapshots. Rebuild Gold before analytics validation.'
)
FROM analytics_validation_context;

## 2. Check that Gold passed its validation

**What:** find the latest Gold validation run for the current project fact. Stop if any of its STOP checks failed for any Gold table, or if it did not check the current population and exposure facts.

**Why:** analytics must never publish numbers from Gold that failed its own checks. Q4 reads the population fact and Q5 reads the exposure fact, so a Gold validation of the project fact alone is not enough: the same run must cover the population and exposure runs the views read now. The run IDs follow `09_validation_gold`, which records the newest `run_id` of each fact as `pipeline_run_id`. Bronze and Silver are not re-checked, because their own validators and the Gold validator already cover them. This is the same gate that each Gold notebook runs on Silver.

**Expected result:** `gold_stop_failures` is 0, `current_population_checks` and `current_exposure_checks` are above 0, the `ASSERT_TRUE` returns `NULL`, and no error is raised.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW analytics_validation_gold_gate AS
WITH latest_gold_validation AS (
    SELECT dq.validation_run_id
    FROM `04-validation`.gold_dq_results AS dq
    INNER JOIN analytics_validation_context AS context
        ON dq.pipeline_run_id = context.project_fact_run_id
    WHERE dq.table_name = 'fact_project_snapshot'
    QUALIFY ROW_NUMBER() OVER (ORDER BY dq.run_ts DESC) = 1
),

current_facts AS (
    -- the same run ids 09_validation_gold records for these facts
    SELECT
        (SELECT MAX(run_id) FROM `03-gold`.fact_region_population) AS population_run_id,
        (SELECT MAX(run_id) FROM `03-gold`.fact_region_flood_exposure) AS exposure_run_id
)

SELECT
    COUNT(*) AS gold_checks,
    COUNT_IF(dq.action = 'stop' AND dq.status IN ('FAIL', 'ERROR')) AS gold_stop_failures,
    COUNT_IF(dq.table_name = 'fact_region_population' AND dq.pipeline_run_id = facts.population_run_id)
        AS current_population_checks,
    COUNT_IF(dq.table_name = 'fact_region_flood_exposure' AND dq.pipeline_run_id = facts.exposure_run_id)
        AS current_exposure_checks,
    MAX(dq.validation_run_id) AS gold_validation_run_id,
    MAX(dq.rule_version) AS gold_rule_version
FROM `04-validation`.gold_dq_results AS dq
INNER JOIN latest_gold_validation AS latest
    ON dq.validation_run_id = latest.validation_run_id
CROSS JOIN current_facts AS facts;

SELECT * FROM analytics_validation_gold_gate;

SELECT ASSERT_TRUE(
    gold_checks > 0
    AND gold_stop_failures = 0
    AND current_population_checks > 0
    AND current_exposure_checks > 0,
    'STOP: Gold validation is missing, failed, or does not cover the current project, population and exposure facts. Run 04_validation/09_validation_gold first.'
)
FROM analytics_validation_gold_gate;

## 3. Q1: recompute regional investment from Gold

**What:** rebuild the Q1 numbers for each official region straight from Gold, then compare them with `analytics_regional_investment` region by region.

**Why:** the view and this check use different code. The check counts rows with `COUNT(*)` and stops separately if a project has more than one current row, while the view counts distinct projects. Budgets are compared with `<=>`, so a `NULL` budget that turned into `0` is a break. Q3, Q4, and Q5 take their regions from Q1, so Q1 is checked against Gold first.

**Expected result:** one compared row per official region, and no break columns set to `true`.

In [ ]:
%sql
-- the view under test, from the analytics schema
CREATE OR REPLACE TEMPORARY VIEW analytics_validation_q1_output AS
SELECT * FROM `05-analytics`.analytics_regional_investment;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_q1_current AS
SELECT
    fact.project_key,
    fact.region_key,
    fact.region_match_status,
    fact.reported_budget_pesos,
    fact.coordinate_status,
    region.region_key IS NOT NULL AS region_found,
    region.psgc_version
FROM `03-gold`.fact_project_snapshot AS fact
LEFT JOIN `03-gold`.dim_region AS region
    ON fact.region_key = region.region_key
WHERE fact.is_current_snapshot;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_q1_expected AS
WITH psgc_versions AS (
    SELECT DISTINCT psgc_version
    FROM analytics_validation_q1_current
    WHERE region_key <> 0 AND region_found
),

regions AS (
    SELECT
        region.region_key,
        region.region_name,
        region.is_barmm
    FROM `03-gold`.dim_region AS region
    INNER JOIN psgc_versions AS version
        ON region.psgc_version <=> version.psgc_version
    WHERE region.region_key <> 0
),

projects AS (
    SELECT
        region_key,
        COUNT(*) AS project_count,
        COUNT_IF(reported_budget_pesos IS NOT NULL) AS projects_with_budget,
        COUNT_IF(coordinate_status = 'VALID_PAIR') AS projects_with_coordinates,
        COUNT_IF(region_match_status = 'MATCHED_WITH_CONFLICT') AS projects_matched_with_conflict,
        SUM(reported_budget_pesos) AS total_reported_budget_pesos
    FROM analytics_validation_q1_current
    WHERE region_key <> 0 AND region_found
    GROUP BY ALL
),

totals AS (
    SELECT
        regions.region_key,
        regions.region_name,
        regions.is_barmm,
        COALESCE(projects.project_count, 0) AS project_count,
        COALESCE(projects.projects_with_budget, 0) AS projects_with_budget,
        COALESCE(projects.projects_with_coordinates, 0) AS projects_with_coordinates,
        COALESCE(projects.projects_matched_with_conflict, 0) AS projects_matched_with_conflict,
        projects.total_reported_budget_pesos,
        CASE
            WHEN COALESCE(projects.project_count, 0) = 0 THEN no_project_coverage_status
            WHEN regions.is_barmm THEN limited_coverage_status
            ELSE reported_coverage_status
        END AS source_coverage_status
    FROM regions
    LEFT JOIN projects
        ON regions.region_key = projects.region_key
)

SELECT
    totals.*,
    ROUND(100.0 * projects_with_budget / NULLIF(project_count, 0), 2) AS budget_coverage_pct,
    ROUND(100.0 * projects_with_coordinates / NULLIF(project_count, 0), 2) AS coordinate_coverage_pct,
    ROUND(total_reported_budget_pesos / NULLIF(projects_with_budget, 0), 2) AS avg_budget_per_project_pesos,
    ROUND(
        100.0 * total_reported_budget_pesos / NULLIF(SUM(total_reported_budget_pesos) OVER (), 0), 4
    ) AS share_pct,
    CASE WHEN source_coverage_status = reported_coverage_status THEN RANK() OVER (
        PARTITION BY source_coverage_status = reported_coverage_status
        ORDER BY total_reported_budget_pesos DESC NULLS LAST
    ) END AS budget_rank,
    CASE WHEN source_coverage_status = reported_coverage_status THEN RANK() OVER (
        PARTITION BY source_coverage_status = reported_coverage_status
        ORDER BY project_count DESC
    ) END AS project_count_rank
FROM totals;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_q1_compare AS
SELECT
    COALESCE(expected.region_key, output.region_key) AS region_key,
    COALESCE(expected.region_name, output.region_name) AS region_name,
    expected.is_barmm,
    output.source_coverage_status,
    output.project_count,
    output.total_reported_budget_pesos AS output_budget_pesos,
    expected.region_key IS NULL AS extra_in_output,
    output.region_key IS NULL AS missing_from_output,
    NOT (output.project_count <=> expected.project_count) AS project_count_break,
    NOT (output.projects_with_budget <=> expected.projects_with_budget) AS projects_with_budget_break,
    NOT (output.total_reported_budget_pesos <=> expected.total_reported_budget_pesos) AS budget_break,
    NOT (output.budget_coverage_pct <=> expected.budget_coverage_pct) AS coverage_pct_break,
    NOT (output.projects_with_coordinates <=> expected.projects_with_coordinates)
    OR NOT (output.coordinate_coverage_pct <=> expected.coordinate_coverage_pct)
    OR NOT (output.projects_matched_with_conflict <=> expected.projects_matched_with_conflict) AS coordinate_break,
    NOT (output.avg_budget_per_project_pesos <=> expected.avg_budget_per_project_pesos) AS average_break,
    NOT (output.share_of_included_regions_budget_pct <=> expected.share_pct) AS share_break,
    NOT (output.source_coverage_status <=> expected.source_coverage_status) AS status_break,
    NOT (output.budget_rank <=> expected.budget_rank)
    OR NOT (output.project_count_rank <=> expected.project_count_rank) AS rank_break,
    NOT (output.project_vs_budget_rank_gap <=> expected.project_count_rank - expected.budget_rank) AS rank_gap_break,
    COALESCE(output.project_count, 0) = 0 AND output.total_reported_budget_pesos IS NOT NULL AS budget_without_projects
FROM analytics_validation_q1_expected AS expected
FULL OUTER JOIN analytics_validation_q1_output AS output
    ON expected.region_key = output.region_key;

SELECT * FROM analytics_validation_q1_compare ORDER BY region_name;

### Projects that Q1 leaves out

**What:** count the current projects on region key `0`, and any whose region key has no `dim_region` row, by Silver match status.

**Why:** key `0` covers Central Office, unmatched, and ambiguous projects, not only Central Office. Q1 does not rank them, so their projects and budget must stay visible here.

**Expected result:** a short table by match status, and no row with the reason `region key not in dim_region`.

In [ ]:
%sql
SELECT
    CASE WHEN region_key = 0 THEN 'region key 0' ELSE 'region key not in dim_region' END AS exclusion_reason,
    region_match_status,
    COUNT(*) AS projects,
    SUM(reported_budget_pesos) AS reported_budget_pesos,
    COUNT_IF(reported_budget_pesos IS NULL) AS projects_without_budget,
    COUNT_IF(reported_budget_pesos = 0) AS projects_with_zero_budget
FROM analytics_validation_q1_current
WHERE region_key = 0 OR NOT region_found
GROUP BY ALL
ORDER BY projects DESC;

## 4. Q2: recompute the category portfolio from Gold

**What:** rebuild each category row from Gold and compare it with `analytics_category_portfolio`. Then check that the categories add up to every current Gold project and budget.

**Why:** Q2 is national. It counts Central Office and unmatched projects too, so its total must equal the full Gold budget, not the Q1 total. Two category facts stay visible as FLAG rows: projects with no usable category, and Multi-sector projects that count in the flood control share because one of their components is Flood Control and Drainage (D-39).

**Expected result:** one compared row per category combination, and no break columns set to `true`.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW analytics_validation_q2_output AS
SELECT * FROM `05-analytics`.analytics_category_portfolio;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_q2_expected AS
WITH totals AS (
    SELECT
        project.standardized_sector,
        project.is_dpwh_flood_related,
        project.infra_type_mapping_status,
        COUNT(*) AS project_count,
        COUNT_IF(fact.reported_budget_pesos IS NOT NULL) AS projects_with_budget,
        SUM(fact.reported_budget_pesos) AS total_reported_budget_pesos
    FROM `03-gold`.fact_project_snapshot AS fact
    LEFT JOIN `03-gold`.dim_project AS project
        ON fact.project_key = project.project_key
    WHERE fact.is_current_snapshot
    GROUP BY ALL
)

SELECT
    totals.*,
    ROUND(total_reported_budget_pesos / NULLIF(projects_with_budget, 0), 2) AS avg_budget_per_project_pesos,
    ROUND(
        100.0 * total_reported_budget_pesos / NULLIF(SUM(total_reported_budget_pesos) OVER (), 0), 4
    ) AS share_pct,
    ROUND(
        100.0 * SUM(CASE WHEN is_dpwh_flood_related THEN total_reported_budget_pesos ELSE 0 END) OVER ()
        / NULLIF(SUM(total_reported_budget_pesos) OVER (), 0),
        4
    ) AS flood_share_pct,
    ROUND(
        100.0 * SUM(
            CASE WHEN standardized_sector = 'Flood Control and Drainage' THEN total_reported_budget_pesos ELSE 0 END
        ) OVER ()
        / NULLIF(SUM(total_reported_budget_pesos) OVER (), 0),
        4
    ) AS single_flood_share_pct
FROM totals;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_q2_compare AS
SELECT
    COALESCE(expected.standardized_sector, output.standardized_sector) AS standardized_sector,
    -- the sector itself can be null (d-37), so a missing side is found by its project count
    expected.project_count IS NULL AS extra_in_output,
    output.project_count IS NULL AS missing_from_output,
    expected.project_count IS NOT NULL AND expected.standardized_sector IS NULL AS is_unmapped,
    NOT (output.project_count <=> expected.project_count)
    OR NOT (output.projects_with_budget <=> expected.projects_with_budget)
    OR NOT (output.total_reported_budget_pesos <=> expected.total_reported_budget_pesos) AS count_or_budget_break,
    NOT (output.avg_budget_per_project_pesos <=> expected.avg_budget_per_project_pesos) AS average_break,
    NOT (output.share_of_total_budget_pct <=> expected.share_pct) AS share_break,
    NOT (output.flood_control_share_of_portfolio_pct <=> expected.flood_share_pct)
    OR NOT (output.flood_control_single_sector_share_pct <=> expected.single_flood_share_pct) AS flood_share_break,
    expected.standardized_sector = 'Multi-sector' AND expected.is_dpwh_flood_related AS is_multi_sector_flood,
    expected.project_count AS expected_projects,
    expected.total_reported_budget_pesos AS expected_budget
FROM analytics_validation_q2_expected AS expected
FULL OUTER JOIN analytics_validation_q2_output AS output
    ON expected.standardized_sector <=> output.standardized_sector
    AND expected.is_dpwh_flood_related <=> output.is_dpwh_flood_related
    AND expected.infra_type_mapping_status <=> output.infra_type_mapping_status;

SELECT * FROM analytics_validation_q2_compare ORDER BY expected_budget DESC NULLS LAST;

## 5. Q3: recompute delivery and status from Gold

**What:** rebuild each region, category, and status group from Gold and compare it with `analytics_project_delivery_status`. Then check each region against Q1.

**Why:** the rates must divide by the projects where each flag is known, and every official region must keep a row. A region with no mapped project must have exactly one empty row, and no region with projects may have one.

**Expected result:** no break columns set to `true`.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW analytics_validation_q3_output AS
SELECT * FROM `05-analytics`.analytics_project_delivery_status;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_q3_output_groups AS
SELECT *
FROM analytics_validation_q3_output
WHERE project_count > 0;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_q3_expected AS
SELECT
    fact.region_key,
    project.standardized_sector,
    project.infra_type_mapping_status,
    status.status_group,
    CASE
        WHEN status.status_group IS NOT NULL THEN 'MAPPED'
        WHEN status.status_key = 0 THEN 'NO_SOURCE_STATUS'
        ELSE 'NO_APPROVED_STATUS_RULE'
    END AS status_mapping_status,
    COUNT(*) AS project_count,
    COUNT(fact.long_running_flag) AS long_running_evaluated_count,
    COUNT_IF(fact.long_running_flag) AS long_running_count,
    COUNT(fact.zero_progress_flag) AS zero_progress_evaluated_count,
    COUNT_IF(fact.zero_progress_flag) AS zero_progress_count
FROM `03-gold`.fact_project_snapshot AS fact
INNER JOIN `03-gold`.dim_project AS project
    ON fact.project_key = project.project_key
INNER JOIN `03-gold`.dim_project_status AS status
    ON fact.status_key = status.status_key
WHERE fact.is_current_snapshot
    AND fact.region_key <> 0
GROUP BY ALL;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_q3_compare AS
SELECT
    COALESCE(expected.region_key, output.region_key) AS region_key,
    expected.region_key IS NULL AS extra_in_output,
    output.region_key IS NULL AS missing_from_output,
    NOT (output.project_count <=> expected.project_count)
    OR NOT (output.long_running_evaluated_count <=> expected.long_running_evaluated_count)
    OR NOT (output.long_running_count <=> expected.long_running_count)
    OR NOT (output.zero_progress_evaluated_count <=> expected.zero_progress_evaluated_count)
    OR NOT (output.zero_progress_count <=> expected.zero_progress_count) AS count_break,
    NOT (output.pct_long_running <=> ROUND(
        100.0 * expected.long_running_count / NULLIF(expected.long_running_evaluated_count, 0), 2
    ))
    OR NOT (output.pct_zero_progress <=> ROUND(
        100.0 * expected.zero_progress_count / NULLIF(expected.zero_progress_evaluated_count, 0), 2
    )) AS rate_break,
    expected.region_key IS NOT NULL AND expected.status_group IS NULL AS is_unknown_status,
    expected.project_count AS expected_projects,
    expected.project_count - expected.long_running_evaluated_count AS expected_unevaluated_long_running
FROM analytics_validation_q3_expected AS expected
FULL OUTER JOIN analytics_validation_q3_output_groups AS output
    ON expected.region_key = output.region_key
    AND expected.standardized_sector <=> output.standardized_sector
    AND expected.infra_type_mapping_status <=> output.infra_type_mapping_status
    AND expected.status_group <=> output.status_group
    AND expected.status_mapping_status <=> output.status_mapping_status;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_q3_share_compare AS
SELECT
    output.region_key,
    NOT (output.pct_of_region_projects <=> ROUND(
        100.0 * output.project_count / NULLIF(q1.project_count, 0), 2
    )) AS share_break,
    NOT (output.pct_of_region_sector_projects <=> ROUND(
        100.0 * output.project_count
        / NULLIF(SUM(output.project_count) OVER (PARTITION BY output.region_key, output.standardized_sector), 0),
        2
    )) AS sector_share_break,
    NOT (output.source_coverage_status <=> q1.source_coverage_status) AS status_break
FROM analytics_validation_q3_output AS output
LEFT JOIN analytics_validation_q1_output AS q1
    ON output.region_key = q1.region_key;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_q3_region_compare AS
WITH q3_regions AS (
    SELECT
        region_key,
        SUM(project_count) AS region_projects,
        COUNT_IF(project_count = 0) AS empty_rows,
        COUNT(*) AS region_rows
    FROM analytics_validation_q3_output
    GROUP BY ALL
)

SELECT
    COALESCE(q1.region_key, q3.region_key) AS region_key,
    q1.region_key IS NULL OR q3.region_key IS NULL AS region_set_break,
    NOT (q3.region_projects <=> q1.project_count) AS project_total_break,
    CASE
        WHEN q1.project_count = 0 THEN NOT (COALESCE(q3.empty_rows, 0) = 1 AND COALESCE(q3.region_rows, 0) = 1)
        ELSE COALESCE(q3.empty_rows, 0) > 0
    END AS empty_row_break
FROM analytics_validation_q1_output AS q1
FULL OUTER JOIN q3_regions AS q3
    ON q1.region_key = q3.region_key;

SELECT * FROM analytics_validation_q3_region_compare ORDER BY region_key;

## 6. Q4: recompute population equity from Gold

**What:** rebuild each region from Q1 and the newest primary population in Gold, then compare it with `analytics_population_equity`.

**Why:** both shares in the gap must cover the same `REPORTED` regions, and a region that is not `REPORTED` must show no ratio at all. The Gold validator asks the dashboard datasets to prove that per-person ratios leave region key `0` out ([validation](../../docs/validation.md)). That check lives here.

**Expected result:** no break columns set to `true`.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW analytics_validation_q4_output AS
SELECT * FROM `05-analytics`.analytics_population_equity;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_q4_expected AS
WITH population AS (
    SELECT
        region_key,
        reference_year,
        population_count
    FROM `03-gold`.fact_region_population
    WHERE is_primary_source
        AND region_key <> 0
    QUALIFY reference_year = MAX(reference_year) OVER ()
),

combined AS (
    SELECT
        q1.region_key,
        q1.source_coverage_status,
        q1.total_reported_budget_pesos,
        population.reference_year,
        population.population_count,
        CASE
            WHEN q1.source_coverage_status = reported_coverage_status
                AND q1.total_reported_budget_pesos IS NOT NULL
                AND population.population_count IS NOT NULL
                THEN q1.total_reported_budget_pesos
        END AS comparable_budget,
        CASE
            WHEN q1.source_coverage_status = reported_coverage_status
                AND q1.total_reported_budget_pesos IS NOT NULL
                AND population.population_count IS NOT NULL
                THEN population.population_count
        END AS comparable_population
    FROM analytics_validation_q1_output AS q1
    LEFT JOIN population
        ON q1.region_key = population.region_key
)

SELECT
    combined.*,
    ROUND(100.0 * comparable_budget / NULLIF(SUM(comparable_budget) OVER (), 0), 4) AS investment_share_pct,
    ROUND(100.0 * population_count / NULLIF(SUM(population_count) OVER (), 0), 4) AS population_share_pct,
    ROUND(
        100.0 * comparable_population / NULLIF(SUM(comparable_population) OVER (), 0), 4
    ) AS comparable_population_share_pct,
    ROUND(comparable_budget / NULLIF(comparable_population, 0), 2) AS investment_per_capita_pesos
FROM combined;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_q4_compare AS
SELECT
    COALESCE(expected.region_key, output.region_key) AS region_key,
    output.region_name,
    COALESCE(output.source_coverage_status, expected.source_coverage_status) AS source_coverage_status,
    expected.region_key IS NULL OR output.region_key IS NULL AS region_set_break,
    NOT (output.source_coverage_status <=> expected.source_coverage_status)
    OR NOT (output.total_reported_budget_pesos <=> expected.total_reported_budget_pesos) AS q1_break,
    NOT (output.population_reference_year <=> expected.reference_year)
    OR NOT (output.population_count <=> expected.population_count) AS population_break,
    NOT (output.investment_share_pct <=> expected.investment_share_pct)
    OR NOT (output.population_share_pct <=> expected.population_share_pct)
    OR NOT (output.comparable_population_share_pct <=> expected.comparable_population_share_pct) AS share_break,
    NOT (output.equity_gap_pp <=> expected.investment_share_pct - expected.comparable_population_share_pct) AS gap_break,
    NOT (output.investment_per_capita_pesos <=> expected.investment_per_capita_pesos) AS per_capita_break,
    expected.comparable_population IS NULL
    AND (
        output.investment_share_pct IS NOT NULL
        OR output.comparable_population_share_pct IS NOT NULL
        OR output.equity_gap_pp IS NOT NULL
        OR output.investment_per_capita_pesos IS NOT NULL
    ) AS unreported_ratio_break,
    output.population_count IS NULL AS missing_population,
    expected.comparable_population IS NULL AS not_comparable,
    output.investment_share_pct,
    output.population_share_pct,
    output.comparable_population_share_pct
FROM analytics_validation_q4_expected AS expected
FULL OUTER JOIN analytics_validation_q4_output AS output
    ON expected.region_key = output.region_key;

SELECT * FROM analytics_validation_q4_compare ORDER BY region_key;

## 7. Q5: recompute flood control and exposure from Gold

**What:** rebuild flood control projects and budget, and each region's per-level exposure from the newest Gold exposure run, then compare them with `analytics_flood_control_exposure`.

**Why:** flood levels can overlap, so a total area is allowed only when `level_area_additivity_status` is `ADDITIVE_WITHIN_TOLERANCE` ([Gold model](../../docs/gold_model.md#fact_region_flood_exposure)). A region with no mapped DPWH project must show no flood control value, because the source cannot say. The project-point groups are rebuilt with `COUNT_IF`, and the no-usable-point group as the remainder, so the view's five groups must add up to its project count.

**Expected result:** no break columns set to `true`.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW analytics_validation_q5_output AS
SELECT * FROM `05-analytics`.analytics_flood_control_exposure;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_q5_expected AS
WITH flood AS (
    SELECT
        fact.region_key,
        COUNT(*) AS flood_projects,
        SUM(fact.reported_budget_pesos) AS flood_budget,
        COUNT_IF(fact.flood_match_status = 'MATCHED'
            AND ARRAY_CONTAINS(high_flood_levels, level.flood_susceptibility_level)) AS points_high,
        COUNT_IF(fact.flood_match_status = 'MATCHED'
            AND ARRAY_CONTAINS(low_flood_levels, level.flood_susceptibility_level)) AS points_low,
        COUNT_IF(fact.flood_match_status = 'AMBIGUOUS') AS points_ambiguous,
        COUNT_IF(fact.flood_match_status = 'UNMATCHED' AND fact.coordinate_status = 'VALID_PAIR') AS points_outside
    FROM `03-gold`.fact_project_snapshot AS fact
    INNER JOIN `03-gold`.dim_project AS project
        ON fact.project_key = project.project_key
    LEFT JOIN `03-gold`.dim_flood_susceptibility AS level
        ON fact.flood_susceptibility_key = level.flood_susceptibility_key
    WHERE fact.is_current_snapshot
        AND fact.region_key <> 0
        AND project.is_dpwh_flood_related
    GROUP BY ALL
),

exposure AS (
    SELECT
        exposure.region_key,
        COUNT_IF(exposure.level_area_additivity_status = 'NOT_EVALUATED') AS not_evaluated_rows,
        COUNT_IF(NOT exposure.level_area_additivity_status <=> additive_exposure_status) AS non_additive_rows,
        MAX(CASE WHEN level.flood_susceptibility_level = 'Low' THEN exposure.susceptible_area_sqkm END) AS low_area_sqkm,
        MAX(CASE WHEN level.flood_susceptibility_level = 'Moderate' THEN exposure.susceptible_area_sqkm END)
            AS moderate_area_sqkm,
        MAX(CASE WHEN level.flood_susceptibility_level = 'High' THEN exposure.susceptible_area_sqkm END) AS high_area_sqkm,
        MAX(CASE WHEN level.flood_susceptibility_level = 'Very High' THEN exposure.susceptible_area_sqkm END)
            AS very_high_area_sqkm,
        MAX(CASE WHEN level.flood_susceptibility_level = 'High' THEN exposure.share_of_region_area_pct END)
            AS high_share_of_region_area_pct,
        MAX(CASE WHEN level.flood_susceptibility_level = 'Very High' THEN exposure.share_of_region_area_pct END)
            AS very_high_share_of_region_area_pct,
        SUM(exposure.susceptible_area_sqkm) AS summed_area_sqkm
    FROM `03-gold`.fact_region_flood_exposure AS exposure
    INNER JOIN analytics_validation_context AS context
        ON exposure.run_id = context.exposure_fact_run_id
    INNER JOIN `03-gold`.dim_flood_susceptibility AS level
        ON exposure.flood_susceptibility_key = level.flood_susceptibility_key
    WHERE exposure.flood_susceptibility_key <> 0
    GROUP BY ALL
)

SELECT
    q1.region_key,
    q1.source_coverage_status,
    q1.total_reported_budget_pesos,
    CASE WHEN q1.source_coverage_status <> no_project_coverage_status THEN COALESCE(flood.flood_projects, 0) END
        AS flood_control_project_count,
    flood.flood_budget AS flood_control_budget_pesos,
    CASE WHEN q1.source_coverage_status <> no_project_coverage_status THEN COALESCE(flood.points_high, 0) END
        AS points_high,
    CASE WHEN q1.source_coverage_status <> no_project_coverage_status THEN COALESCE(flood.points_low, 0) END
        AS points_low,
    CASE WHEN q1.source_coverage_status <> no_project_coverage_status THEN COALESCE(flood.points_ambiguous, 0) END
        AS points_ambiguous,
    CASE WHEN q1.source_coverage_status <> no_project_coverage_status THEN COALESCE(flood.points_outside, 0) END
        AS points_outside,
    CASE
        WHEN q1.source_coverage_status <> no_project_coverage_status THEN COALESCE(
            flood.flood_projects - flood.points_high - flood.points_low - flood.points_ambiguous - flood.points_outside, 0
        )
    END AS points_no_usable,
    CASE
        WHEN exposure.region_key IS NULL THEN 'NO_EXPOSURE_DATA'
        WHEN exposure.not_evaluated_rows > 0 THEN 'NOT_EVALUATED'
        WHEN exposure.non_additive_rows > 0 THEN 'NON_ADDITIVE_CROSS_LEVEL_OVERLAP'
        ELSE additive_exposure_status
    END AS exposure_area_status,
    exposure.low_area_sqkm,
    exposure.moderate_area_sqkm,
    exposure.high_area_sqkm,
    exposure.very_high_area_sqkm,
    exposure.high_share_of_region_area_pct,
    exposure.very_high_share_of_region_area_pct,
    exposure.summed_area_sqkm
FROM analytics_validation_q1_output AS q1
LEFT JOIN flood
    ON q1.region_key = flood.region_key
LEFT JOIN exposure
    ON q1.region_key = exposure.region_key;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_q5_compare AS
SELECT
    COALESCE(expected.region_key, output.region_key) AS region_key,
    output.region_name,
    COALESCE(output.exposure_area_status, expected.exposure_area_status) AS exposure_area_status,
    expected.region_key IS NULL OR output.region_key IS NULL AS region_set_break,
    NOT (output.source_coverage_status <=> expected.source_coverage_status)
    OR NOT (output.total_reported_budget_pesos <=> expected.total_reported_budget_pesos) AS q1_break,
    NOT (output.flood_control_project_count <=> expected.flood_control_project_count)
    OR NOT (output.flood_control_budget_pesos <=> expected.flood_control_budget_pesos) AS flood_break,
    NOT (output.flood_control_share_of_region_budget_pct <=> CASE
        WHEN expected.source_coverage_status <> no_project_coverage_status THEN ROUND(
            100.0 * COALESCE(expected.flood_control_budget_pesos, 0)
            / NULLIF(expected.total_reported_budget_pesos, 0), 4
        )
    END) AS flood_share_break,
    NOT (output.exposure_area_status <=> expected.exposure_area_status) AS exposure_status_break,
    NOT ARRAY_CONTAINS(expected_exposure_statuses, output.exposure_area_status) AS invalid_status,
    NOT (output.low_area_sqkm <=> expected.low_area_sqkm)
    OR NOT (output.moderate_area_sqkm <=> expected.moderate_area_sqkm)
    OR NOT (output.high_area_sqkm <=> expected.high_area_sqkm)
    OR NOT (output.very_high_area_sqkm <=> expected.very_high_area_sqkm)
    OR NOT (output.high_share_of_region_area_pct <=> expected.high_share_of_region_area_pct)
    OR NOT (output.very_high_share_of_region_area_pct <=> expected.very_high_share_of_region_area_pct) AS level_area_break,
    output.total_susceptible_area_sqkm IS NOT NULL
    AND NOT output.exposure_area_status <=> additive_exposure_status AS non_additive_total_break,
    NOT (output.total_susceptible_area_sqkm <=> CASE
        WHEN expected.exposure_area_status = additive_exposure_status THEN expected.summed_area_sqkm
    END) AS total_area_break,
    NOT (output.flood_control_pesos_per_sqkm_exposed <=> CASE
        WHEN expected.source_coverage_status <> no_project_coverage_status THEN ROUND(
            COALESCE(expected.flood_control_budget_pesos, 0) / NULLIF(CASE
                WHEN expected.exposure_area_status = additive_exposure_status THEN expected.summed_area_sqkm
            END, 0), 2
        )
    END) AS pesos_per_sqkm_break,
    NOT (output.flood_control_points_high_or_very_high <=> expected.points_high)
    OR NOT (output.flood_control_points_low_or_moderate <=> expected.points_low)
    OR NOT (output.flood_control_points_ambiguous <=> expected.points_ambiguous)
    OR NOT (output.flood_control_points_outside_mapped_zones <=> expected.points_outside)
    OR NOT (output.flood_control_points_no_usable_point <=> expected.points_no_usable)
    OR NOT (output.flood_control_points_classified_pct <=> ROUND(
        100.0 * (expected.points_high + expected.points_low) / NULLIF(expected.flood_control_project_count, 0), 4
    ))
    OR NOT (output.flood_control_points_high_share_of_classified_pct <=> ROUND(
        100.0 * expected.points_high / NULLIF(expected.points_high + expected.points_low, 0), 4
    )) AS point_break,
    NOT (output.flood_control_points_high_or_very_high + output.flood_control_points_low_or_moderate
        + output.flood_control_points_ambiguous + output.flood_control_points_outside_mapped_zones
        + output.flood_control_points_no_usable_point <=> output.flood_control_project_count) AS point_sum_break,
    output.flood_control_points_outside_mapped_zones,
    output.flood_control_points_no_usable_point,
    output.flood_control_project_count,
    output.source_coverage_status = no_project_coverage_status
    AND (
        output.flood_control_project_count IS NOT NULL
        OR output.flood_control_share_of_region_budget_pct IS NOT NULL
        OR output.flood_control_pesos_per_sqkm_exposed IS NOT NULL
        OR output.flood_control_points_classified_pct IS NOT NULL
        OR output.flood_control_points_no_usable_point IS NOT NULL
    ) AS unknown_shown_as_value_break
FROM analytics_validation_q5_expected AS expected
FULL OUTER JOIN analytics_validation_q5_output AS output
    ON expected.region_key = output.region_key;

SELECT * FROM analytics_validation_q5_compare ORDER BY region_key;

## 8. Q6: recompute investment by area and type from Gold

**What:** rebuild each region and sector from Gold and compare it with `analytics_region_category`. Then check each region against Q1.

**Why:** each region's sectors must add up to its Q1 projects and budget, and both shares must divide by the right total. A region with no mapped project must keep exactly one empty row.

**Expected result:** no break columns set to `true`.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW analytics_validation_q6_output AS
SELECT * FROM `05-analytics`.analytics_region_category;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_q6_output_groups AS
SELECT *
FROM analytics_validation_q6_output
WHERE project_count > 0;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_q6_expected AS
WITH sectors AS (
    SELECT
        fact.region_key,
        project.standardized_sector,
        project.infra_type_mapping_status,
        COUNT(*) AS project_count,
        COUNT_IF(fact.reported_budget_pesos IS NOT NULL) AS projects_with_budget,
        SUM(fact.reported_budget_pesos) AS total_reported_budget_pesos
    FROM `03-gold`.fact_project_snapshot AS fact
    INNER JOIN `03-gold`.dim_project AS project
        ON fact.project_key = project.project_key
    INNER JOIN analytics_validation_q1_output AS q1
        ON fact.region_key = q1.region_key
    WHERE fact.is_current_snapshot
    GROUP BY ALL
)

SELECT
    sectors.*,
    ROUND(total_reported_budget_pesos / NULLIF(projects_with_budget, 0), 2) AS avg_budget_per_project_pesos,
    ROUND(
        100.0 * total_reported_budget_pesos
        / NULLIF(SUM(total_reported_budget_pesos) OVER (PARTITION BY region_key), 0),
        4
    ) AS share_of_region_budget_pct,
    ROUND(
        100.0 * total_reported_budget_pesos
        / NULLIF(SUM(total_reported_budget_pesos) OVER (PARTITION BY standardized_sector), 0),
        4
    ) AS share_of_sector_budget_pct
FROM sectors;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_q6_compare AS
SELECT
    COALESCE(expected.region_key, output.region_key) AS region_key,
    expected.region_key IS NULL AS extra_in_output,
    output.region_key IS NULL AS missing_from_output,
    NOT (output.project_count <=> expected.project_count)
    OR NOT (output.projects_with_budget <=> expected.projects_with_budget)
    OR NOT (output.total_reported_budget_pesos <=> expected.total_reported_budget_pesos) AS count_or_budget_break,
    NOT (output.avg_budget_per_project_pesos <=> expected.avg_budget_per_project_pesos) AS average_break,
    NOT (output.share_of_region_budget_pct <=> expected.share_of_region_budget_pct)
    OR NOT (output.share_of_sector_budget_pct <=> expected.share_of_sector_budget_pct) AS share_break
FROM analytics_validation_q6_expected AS expected
FULL OUTER JOIN analytics_validation_q6_output_groups AS output
    ON expected.region_key = output.region_key
    AND expected.standardized_sector <=> output.standardized_sector
    AND expected.infra_type_mapping_status <=> output.infra_type_mapping_status;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_q6_region_compare AS
WITH q6_regions AS (
    SELECT
        region_key,
        MAX(source_coverage_status) AS source_coverage_status,
        SUM(project_count) AS region_projects,
        SUM(total_reported_budget_pesos) AS region_budget,
        COUNT_IF(project_count = 0) AS empty_rows,
        COUNT(*) AS region_rows
    FROM analytics_validation_q6_output
    GROUP BY ALL
)

SELECT
    COALESCE(q1.region_key, q6.region_key) AS region_key,
    q1.region_key IS NULL OR q6.region_key IS NULL AS region_set_break,
    NOT (q6.region_projects <=> q1.project_count)
    OR NOT (q6.region_budget <=> q1.total_reported_budget_pesos)
    OR NOT (q6.source_coverage_status <=> q1.source_coverage_status) AS q1_break,
    CASE
        WHEN q1.project_count = 0 THEN NOT (COALESCE(q6.empty_rows, 0) = 1 AND COALESCE(q6.region_rows, 0) = 1)
        ELSE COALESCE(q6.empty_rows, 0) > 0
    END AS empty_row_break
FROM analytics_validation_q1_output AS q1
FULL OUTER JOIN q6_regions AS q6
    ON q1.region_key = q6.region_key;

SELECT * FROM analytics_validation_q6_region_compare ORDER BY region_key;

## 9. Infrastructure-year trend: recompute from Gold

**What:** build one current Gold row per project with the dimension attributes the exploratory views read. Then rebuild `analytics_infrastructure_year_trend` year by year and compare it with the view. The last query lists every observed `infra_year` with its project count.

**Why:**

- The Gold rows use left joins, so a project without a `dim_project` or `dim_project_status` row stays visible and breaks the totals, instead of dropping out quietly.
- Years are compared with `<=>`, so the `NULL` year is checked like any other year. The years, including `NULL`, must add up to every current Gold project and budget.
- The check counts rows with `COUNT_IF` and `COUNT(*)`, while the view counts distinct projects. The Q1 check "At most one current row per project" stops the run if a project has two current rows, so no project can count twice.
- The flag denominators are rebuilt as `COUNT_IF(flag IS NOT NULL)`, on their own, and each rate is recomputed from them.
- The year list is the evidence for shifted or outlier years. The view keeps every year, and so does the check.

**Expected result:** no break columns set to `true`, and a list of observed years.

In [ ]:
%sql
-- one current gold row per project with the attributes the exploratory views read.
-- left joins, so a project without a dimension row stays visible and breaks the totals
CREATE OR REPLACE TEMPORARY VIEW analytics_validation_exploratory_current AS
SELECT
    fact.project_key,
    fact.region_key,
    fact.status_key,
    fact.reported_budget_pesos,
    fact.long_running_flag,
    fact.zero_progress_flag,
    project.project_key IS NOT NULL AS project_found,
    project.infra_year,
    project.standardized_sector,
    project.infra_type_mapping_status,
    project.implementing_office,
    status.status_key IS NOT NULL AS status_found,
    status.status_group,
    status.source_status
FROM `03-gold`.fact_project_snapshot AS fact
LEFT JOIN `03-gold`.dim_project AS project
    ON fact.project_key = project.project_key
LEFT JOIN `03-gold`.dim_project_status AS status
    ON fact.status_key = status.status_key
WHERE fact.is_current_snapshot;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_year_trend_output AS
SELECT * FROM `05-analytics`.analytics_infrastructure_year_trend;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_year_trend_expected AS
SELECT
    infra_year,
    COUNT(*) AS project_count,
    COUNT_IF(reported_budget_pesos IS NOT NULL) AS projects_with_budget,
    SUM(reported_budget_pesos) AS total_reported_budget_pesos,
    COUNT_IF(status_group = finished_status_group) AS finished_count,
    COUNT_IF(status_group = active_status_group) AS active_count,
    COUNT_IF(status_group = inactive_status_group) AS inactive_count,
    COUNT_IF(status_group IS NULL) AS unmapped_status_count,
    COUNT_IF(source_status = for_procurement_source_status) AS for_procurement_count,
    COUNT_IF(long_running_flag IS NOT NULL) AS long_running_evaluated_count,
    COUNT_IF(long_running_flag) AS long_running_count,
    COUNT_IF(zero_progress_flag IS NOT NULL) AS zero_progress_evaluated_count,
    COUNT_IF(zero_progress_flag) AS zero_progress_count
FROM analytics_validation_exploratory_current
GROUP BY ALL;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_year_trend_compare AS
SELECT
    COALESCE(expected.infra_year, output.infra_year) AS infra_year,
    -- the year itself can be null, so a missing side is found by its project count
    expected.project_count IS NULL AS extra_in_output,
    output.project_count IS NULL AS missing_from_output,
    NOT (output.project_count <=> expected.project_count)
    OR NOT (output.projects_with_budget <=> expected.projects_with_budget)
    OR NOT (output.total_reported_budget_pesos <=> expected.total_reported_budget_pesos) AS count_or_budget_break,
    NOT (output.budget_coverage_pct <=> ROUND(
        100.0 * expected.projects_with_budget / NULLIF(expected.project_count, 0), 2
    ))
    OR NOT (output.avg_budget_per_project_pesos <=> ROUND(
        expected.total_reported_budget_pesos / NULLIF(expected.projects_with_budget, 0), 2
    )) AS average_break,
    NOT (output.finished_count <=> expected.finished_count)
    OR NOT (output.active_count <=> expected.active_count)
    OR NOT (output.inactive_count <=> expected.inactive_count)
    OR NOT (output.unmapped_status_count <=> expected.unmapped_status_count) AS status_break,
    NOT (output.finished_count + output.active_count + output.inactive_count + output.unmapped_status_count
        <=> output.project_count) AS status_sum_break,
    NOT (output.for_procurement_count <=> expected.for_procurement_count) AS for_procurement_break,
    NOT (output.long_running_evaluated_count <=> expected.long_running_evaluated_count)
    OR NOT (output.long_running_count <=> expected.long_running_count) AS long_running_break,
    NOT (output.zero_progress_evaluated_count <=> expected.zero_progress_evaluated_count)
    OR NOT (output.zero_progress_count <=> expected.zero_progress_count) AS zero_progress_break,
    NOT (output.pct_long_running <=> ROUND(
        100.0 * expected.long_running_count / NULLIF(expected.long_running_evaluated_count, 0), 2
    ))
    OR NOT (output.pct_zero_progress <=> ROUND(
        100.0 * expected.zero_progress_count / NULLIF(expected.zero_progress_evaluated_count, 0), 2
    )) AS rate_break,
    output.long_running_count > output.long_running_evaluated_count
    OR output.long_running_evaluated_count > output.project_count
    OR output.zero_progress_count > output.zero_progress_evaluated_count
    OR output.zero_progress_evaluated_count > output.project_count AS impossible_row
FROM analytics_validation_year_trend_expected AS expected
FULL OUTER JOIN analytics_validation_year_trend_output AS output
    ON expected.infra_year <=> output.infra_year;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_year_trend_metrics AS
WITH output_metrics AS (
    SELECT
        COUNT(*) AS yt_output_rows,
        COUNT(DISTINCT NAMED_STRUCT('year', infra_year)) AS yt_output_keys,
        COALESCE(SUM(project_count), 0) AS yt_output_projects,
        COALESCE(SUM(total_reported_budget_pesos), 0) AS yt_output_budget
    FROM analytics_validation_year_trend_output
),

gold_metrics AS (
    SELECT
        COUNT(*) AS yt_gold_rows,
        COUNT(DISTINCT project_key) AS yt_gold_projects,
        COALESCE(SUM(reported_budget_pesos), 0) AS yt_gold_budget,
        COUNT_IF(NOT project_found OR NOT status_found) AS yt_unjoined_projects,
        COUNT_IF(infra_year IS NULL) AS yt_null_year_projects,
        COALESCE(SUM(CASE WHEN infra_year IS NULL THEN reported_budget_pesos END), 0) AS yt_null_year_budget,
        COUNT_IF(long_running_flag IS NULL) AS yt_unevaluated_long_running,
        COUNT_IF(zero_progress_flag) AS yt_zero_progress_projects,
        COUNT_IF(zero_progress_flag AND source_status = for_procurement_source_status)
            AS yt_zero_progress_for_procurement
    FROM analytics_validation_exploratory_current
),

year_metrics AS (
    -- the observed years and their projects, sorted so the detail reads the same on every run
    SELECT
        MIN(infra_year) AS yt_min_year,
        MAX(infra_year) AS yt_max_year,
        ARRAY_JOIN(TRANSFORM(
            SORT_ARRAY(COLLECT_LIST(NAMED_STRUCT('year', infra_year, 'projects', project_count))),
            entry -> CONCAT(COALESCE(CAST(entry.year AS STRING), 'NULL'), '=', CAST(entry.projects AS STRING))
        ), ', ') AS yt_year_detail
    FROM analytics_validation_year_trend_expected
),

compare_metrics AS (
    SELECT
        COUNT(*) AS yt_compared_rows,
        COUNT_IF(extra_in_output OR missing_from_output) AS yt_row_set_breaks,
        COUNT_IF(count_or_budget_break) AS yt_count_budget_breaks,
        COUNT_IF(average_break) AS yt_average_breaks,
        COUNT_IF(status_break) AS yt_status_breaks,
        COUNT_IF(status_sum_break) AS yt_status_sum_breaks,
        COUNT_IF(for_procurement_break) AS yt_for_procurement_breaks,
        COUNT_IF(long_running_break) AS yt_long_running_breaks,
        COUNT_IF(zero_progress_break) AS yt_zero_progress_breaks,
        COUNT_IF(rate_break) AS yt_rate_breaks,
        COUNT_IF(impossible_row) AS yt_impossible_rows
    FROM analytics_validation_year_trend_compare
),

lineage_metrics AS (
    SELECT
        COUNT_IF(NOT output.source_snapshot_id <=> context.project_snapshot_id) AS yt_lineage_breaks
    FROM analytics_validation_year_trend_output AS output
    CROSS JOIN analytics_validation_context AS context
)

SELECT *
FROM output_metrics
CROSS JOIN gold_metrics
CROSS JOIN year_metrics
CROSS JOIN compare_metrics
CROSS JOIN lineage_metrics;

SELECT * FROM analytics_validation_year_trend_compare ORDER BY infra_year NULLS LAST;

-- evidence: every observed infra_year and its projects, so shifted or outlier years stay visible
SELECT
    infra_year,
    project_count,
    total_reported_budget_pesos,
    for_procurement_count,
    unmapped_status_count
FROM analytics_validation_year_trend_expected
ORDER BY infra_year NULLS LAST;

## 10. Infrastructure-year sector mix: recompute from Gold

**What:** rebuild each year and sector row from Gold and compare it with `analytics_infrastructure_year_sector`. Then check each year against the trend view and against all current Gold projects.

**Why:** `share_of_year_budget_pct` must divide by the budget of the same `infra_year`, with the `NULL` year as its own cohort. Each year's shares must add up to 100. Each year must add up to the projects and budget of the same year in `analytics_infrastructure_year_trend`, so the two views always agree.

**Expected result:** no break columns set to `true`.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW analytics_validation_year_sector_output AS
SELECT * FROM `05-analytics`.analytics_infrastructure_year_sector;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_year_sector_expected AS
WITH sectors AS (
    SELECT
        infra_year,
        standardized_sector,
        infra_type_mapping_status,
        COUNT(*) AS project_count,
        COUNT_IF(reported_budget_pesos IS NOT NULL) AS projects_with_budget,
        SUM(reported_budget_pesos) AS total_reported_budget_pesos
    FROM analytics_validation_exploratory_current
    GROUP BY ALL
)

SELECT
    sectors.*,
    ROUND(total_reported_budget_pesos / NULLIF(projects_with_budget, 0), 2) AS avg_budget_per_project_pesos,
    ROUND(
        100.0 * total_reported_budget_pesos
        / NULLIF(SUM(total_reported_budget_pesos) OVER (PARTITION BY infra_year), 0),
        4
    ) AS share_of_year_budget_pct
FROM sectors;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_year_sector_compare AS
SELECT
    COALESCE(expected.infra_year, output.infra_year) AS infra_year,
    expected.project_count IS NULL AS extra_in_output,
    output.project_count IS NULL AS missing_from_output,
    NOT (output.project_count <=> expected.project_count)
    OR NOT (output.projects_with_budget <=> expected.projects_with_budget)
    OR NOT (output.total_reported_budget_pesos <=> expected.total_reported_budget_pesos) AS count_or_budget_break,
    NOT (output.avg_budget_per_project_pesos <=> expected.avg_budget_per_project_pesos) AS average_break,
    NOT (output.share_of_year_budget_pct <=> expected.share_of_year_budget_pct) AS share_break
FROM analytics_validation_year_sector_expected AS expected
FULL OUTER JOIN analytics_validation_year_sector_output AS output
    ON expected.infra_year <=> output.infra_year
    AND expected.standardized_sector <=> output.standardized_sector
    AND expected.infra_type_mapping_status <=> output.infra_type_mapping_status;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_year_sector_year_compare AS
WITH sector_years AS (
    SELECT
        infra_year,
        SUM(project_count) AS year_projects,
        SUM(total_reported_budget_pesos) AS year_budget,
        SUM(share_of_year_budget_pct) AS share_total
    FROM analytics_validation_year_sector_output
    GROUP BY ALL
)

SELECT
    COALESCE(sector_years.infra_year, trend.infra_year) AS infra_year,
    sector_years.year_budget,
    sector_years.share_total,
    -- a missing side is found by its project count, because the year can be null
    sector_years.year_projects IS NULL OR trend.project_count IS NULL
    OR NOT (sector_years.year_projects <=> trend.project_count)
    OR NOT (sector_years.year_budget <=> trend.total_reported_budget_pesos) AS trend_break
FROM sector_years
FULL OUTER JOIN analytics_validation_year_trend_output AS trend
    ON sector_years.infra_year <=> trend.infra_year;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_year_sector_metrics AS
WITH output_metrics AS (
    SELECT
        COUNT(*) AS ys_output_rows,
        COUNT(DISTINCT NAMED_STRUCT(
            'year', infra_year, 'sector', standardized_sector, 'sector_reason', infra_type_mapping_status
        )) AS ys_output_keys,
        COALESCE(SUM(project_count), 0) AS ys_output_projects,
        COALESCE(SUM(total_reported_budget_pesos), 0) AS ys_output_budget
    FROM analytics_validation_year_sector_output
),

compare_metrics AS (
    SELECT
        COUNT(*) AS ys_compared_rows,
        COUNT_IF(extra_in_output OR missing_from_output) AS ys_row_set_breaks,
        COUNT_IF(count_or_budget_break) AS ys_count_budget_breaks,
        COUNT_IF(average_break) AS ys_average_breaks,
        COUNT_IF(share_break) AS ys_share_breaks
    FROM analytics_validation_year_sector_compare
),

year_metrics AS (
    SELECT
        COUNT(*) AS ys_years,
        COUNT_IF(trend_break) AS ys_trend_breaks,
        COUNT_IF(year_budget > 0 AND NOT ABS(share_total - 100) <= share_total_tolerance_pct) AS ys_share_total_breaks
    FROM analytics_validation_year_sector_year_compare
),

gold_metrics AS (
    SELECT
        COUNT(DISTINCT project_key) AS ys_gold_projects,
        COALESCE(SUM(reported_budget_pesos), 0) AS ys_gold_budget
    FROM analytics_validation_exploratory_current
),

lineage_metrics AS (
    SELECT
        COUNT_IF(NOT output.source_snapshot_id <=> context.project_snapshot_id) AS ys_lineage_breaks
    FROM analytics_validation_year_sector_output AS output
    CROSS JOIN analytics_validation_context AS context
)

SELECT *
FROM output_metrics
CROSS JOIN compare_metrics
CROSS JOIN year_metrics
CROSS JOIN gold_metrics
CROSS JOIN lineage_metrics;

SELECT * FROM analytics_validation_year_sector_year_compare ORDER BY infra_year NULLS LAST;

## 11. Delivery by cohort: recompute from Gold

**What:** rebuild each year, status group, and status reason from Gold and compare it with `analytics_delivery_cohort`. Then check each year against the trend view.

**Why:**

- The status reason is rebuilt with the validator's own rule order, the same one the Q3 check uses. A row with a status group must be `MAPPED`, and a row without one must give a reason. So a status key `0` or an unmapped status can never drop out.
- `pct_of_year_projects` is checked against a year total counted straight from Gold, including projects with no status group. The rows of a year must add up to 100. Each row is rounded to 2 decimals, so the allowed gap is 0.005 per row of that year.
- The long-running and zero-progress denominators are rebuilt as `COUNT_IF(flag IS NOT NULL)`, on their own, and must also add up to the trend view's denominators for the same year.

**Expected result:** no break columns set to `true`.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW analytics_validation_delivery_cohort_output AS
SELECT * FROM `05-analytics`.analytics_delivery_cohort;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_delivery_cohort_expected AS
WITH cohort_groups AS (
    SELECT
        infra_year,
        status_group,
        CASE
            WHEN status_group IS NOT NULL THEN 'MAPPED'
            WHEN status_key = 0 THEN 'NO_SOURCE_STATUS'
            ELSE 'NO_APPROVED_STATUS_RULE'
        END AS status_mapping_status,
        COUNT(*) AS project_count,
        COUNT_IF(source_status = for_procurement_source_status) AS for_procurement_count,
        COUNT_IF(long_running_flag IS NOT NULL) AS long_running_evaluated_count,
        COUNT_IF(long_running_flag) AS long_running_count,
        COUNT_IF(zero_progress_flag IS NOT NULL) AS zero_progress_evaluated_count,
        COUNT_IF(zero_progress_flag) AS zero_progress_count
    FROM analytics_validation_exploratory_current
    GROUP BY ALL
),

cohort_years AS (
    -- every project of the year, counted straight from gold, including unmapped statuses
    SELECT
        infra_year,
        COUNT(*) AS year_projects
    FROM analytics_validation_exploratory_current
    GROUP BY ALL
)

SELECT
    cohort_groups.*,
    ROUND(100.0 * cohort_groups.project_count / NULLIF(cohort_years.year_projects, 0), 2) AS pct_of_year_projects
FROM cohort_groups
LEFT JOIN cohort_years
    ON cohort_groups.infra_year <=> cohort_years.infra_year;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_delivery_cohort_compare AS
SELECT
    COALESCE(expected.infra_year, output.infra_year) AS infra_year,
    expected.project_count IS NULL AS extra_in_output,
    output.project_count IS NULL AS missing_from_output,
    NOT (output.project_count <=> expected.project_count)
    OR NOT (output.for_procurement_count <=> expected.for_procurement_count) AS count_break,
    NOT (output.long_running_evaluated_count <=> expected.long_running_evaluated_count)
    OR NOT (output.long_running_count <=> expected.long_running_count) AS long_running_break,
    NOT (output.zero_progress_evaluated_count <=> expected.zero_progress_evaluated_count)
    OR NOT (output.zero_progress_count <=> expected.zero_progress_count) AS zero_progress_break,
    NOT (output.pct_long_running <=> ROUND(
        100.0 * expected.long_running_count / NULLIF(expected.long_running_evaluated_count, 0), 2
    ))
    OR NOT (output.pct_zero_progress <=> ROUND(
        100.0 * expected.zero_progress_count / NULLIF(expected.zero_progress_evaluated_count, 0), 2
    )) AS rate_break,
    NOT (output.pct_of_year_projects <=> expected.pct_of_year_projects) AS year_share_break,
    output.long_running_count > output.long_running_evaluated_count
    OR output.long_running_evaluated_count > output.project_count
    OR output.zero_progress_count > output.zero_progress_evaluated_count
    OR output.zero_progress_evaluated_count > output.project_count AS impossible_row,
    (output.status_group IS NOT NULL AND NOT output.status_mapping_status <=> 'MAPPED')
    OR (
        output.project_count IS NOT NULL AND output.status_group IS NULL
        AND NOT COALESCE(output.status_mapping_status IN ('NO_SOURCE_STATUS', 'NO_APPROVED_STATUS_RULE'), FALSE)
    ) AS reason_break,
    expected.status_group IS NULL AND expected.project_count IS NOT NULL AS is_unmapped_status,
    expected.status_mapping_status AS expected_status_reason,
    expected.project_count AS expected_projects
FROM analytics_validation_delivery_cohort_expected AS expected
FULL OUTER JOIN analytics_validation_delivery_cohort_output AS output
    ON expected.infra_year <=> output.infra_year
    AND expected.status_group <=> output.status_group
    AND expected.status_mapping_status <=> output.status_mapping_status;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_delivery_cohort_year_compare AS
WITH cohort_years AS (
    SELECT
        infra_year,
        COUNT(*) AS year_rows,
        SUM(project_count) AS year_projects,
        SUM(for_procurement_count) AS year_for_procurement,
        SUM(long_running_evaluated_count) AS year_long_running_evaluated,
        SUM(zero_progress_evaluated_count) AS year_zero_progress_evaluated,
        SUM(pct_of_year_projects) AS share_total
    FROM analytics_validation_delivery_cohort_output
    GROUP BY ALL
)

SELECT
    COALESCE(cohort_years.infra_year, trend.infra_year) AS infra_year,
    cohort_years.year_rows,
    cohort_years.share_total,
    cohort_years.year_projects IS NULL OR trend.project_count IS NULL
    OR NOT (cohort_years.year_projects <=> trend.project_count)
    OR NOT (cohort_years.year_for_procurement <=> trend.for_procurement_count)
    OR NOT (cohort_years.year_long_running_evaluated <=> trend.long_running_evaluated_count)
    OR NOT (cohort_years.year_zero_progress_evaluated <=> trend.zero_progress_evaluated_count) AS trend_break,
    -- each row is rounded to 2 decimals, so a year may miss 100 by 0.005 per row
    cohort_years.year_projects > 0
    AND NOT ABS(cohort_years.share_total - 100) <= 0.005 * cohort_years.year_rows AS share_total_break
FROM cohort_years
FULL OUTER JOIN analytics_validation_year_trend_output AS trend
    ON cohort_years.infra_year <=> trend.infra_year;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_delivery_cohort_metrics AS
WITH output_metrics AS (
    SELECT
        COUNT(*) AS dc_output_rows,
        COUNT(DISTINCT NAMED_STRUCT(
            'year', infra_year, 'status', status_group, 'status_reason', status_mapping_status
        )) AS dc_output_keys
    FROM analytics_validation_delivery_cohort_output
),

compare_metrics AS (
    SELECT
        COUNT(*) AS dc_compared_rows,
        COUNT_IF(extra_in_output OR missing_from_output) AS dc_row_set_breaks,
        COUNT_IF(count_break) AS dc_count_breaks,
        COUNT_IF(long_running_break) AS dc_long_running_breaks,
        COUNT_IF(zero_progress_break) AS dc_zero_progress_breaks,
        COUNT_IF(rate_break) AS dc_rate_breaks,
        COUNT_IF(year_share_break) AS dc_year_share_breaks,
        COUNT_IF(impossible_row) AS dc_impossible_rows,
        COUNT_IF(reason_break) AS dc_reason_breaks,
        COALESCE(SUM(expected_projects), 0) AS dc_expected_projects,
        COALESCE(SUM(CASE WHEN is_unmapped_status THEN expected_projects END), 0) AS dc_unmapped_projects,
        COALESCE(SUM(CASE WHEN expected_status_reason = 'NO_SOURCE_STATUS' THEN expected_projects END), 0)
            AS dc_no_source_status_projects,
        COALESCE(SUM(CASE WHEN expected_status_reason = 'NO_APPROVED_STATUS_RULE' THEN expected_projects END), 0)
            AS dc_no_rule_projects
    FROM analytics_validation_delivery_cohort_compare
),

year_metrics AS (
    SELECT
        COUNT(*) AS dc_years,
        COUNT_IF(trend_break) AS dc_trend_breaks,
        COUNT_IF(share_total_break) AS dc_share_total_breaks
    FROM analytics_validation_delivery_cohort_year_compare
),

lineage_metrics AS (
    SELECT
        COUNT_IF(NOT output.source_snapshot_id <=> context.project_snapshot_id) AS dc_lineage_breaks
    FROM analytics_validation_delivery_cohort_output AS output
    CROSS JOIN analytics_validation_context AS context
)

SELECT *
FROM output_metrics
CROSS JOIN compare_metrics
CROSS JOIN year_metrics
CROSS JOIN lineage_metrics;

SELECT * FROM analytics_validation_delivery_cohort_year_compare ORDER BY infra_year NULLS LAST;

## 12. Project size concentration: recompute from Gold

**What:** rebuild each region's budget distribution from Gold and compare it with `analytics_project_size_concentration`. Then check each region against Q1.

**Why:**

- The distribution is rebuilt only from current Gold rows where `reported_budget_pesos IS NOT NULL` and the region key is not `0`. A `NULL` budget is in no median, p90, or average, and a reported 0 stays in. The check uses the same documented definition as the view, the exact `PERCENTILE_CONT`. The independence comes from rebuilding the rows from Gold, not from a different function.
- The zero-budget count is `reported_budget_pesos = 0`, so a missing budget is never counted as 0.
- Regions and coverage must match Q1, so key `0` never enters the comparison. Q1 already counts key `0` projects separately.
- A region with no mapped project must show no median, p90, max, or largest-project share.

**Expected result:** no break columns set to `true`.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW analytics_validation_size_output AS
SELECT * FROM `05-analytics`.analytics_project_size_concentration;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_size_expected AS
WITH known_budgets AS (
    -- only projects with a reported budget. a reported 0 stays in, a null budget is left out
    SELECT
        region_key,
        reported_budget_pesos
    FROM analytics_validation_exploratory_current
    WHERE region_key <> 0
        AND reported_budget_pesos IS NOT NULL
),

distribution AS (
    SELECT
        region_key,
        COUNT(*) AS budget_projects,
        COUNT_IF(reported_budget_pesos = 0) AS zero_budget_projects,
        SUM(reported_budget_pesos) AS budget_total,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY reported_budget_pesos) AS median_budget,
        PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY reported_budget_pesos) AS p90_budget,
        MAX(reported_budget_pesos) AS max_budget
    FROM known_budgets
    GROUP BY ALL
),

gold_regions AS (
    SELECT
        region_key,
        COUNT(*) AS project_count
    FROM analytics_validation_exploratory_current
    WHERE region_key <> 0
    GROUP BY ALL
)

SELECT
    q1.region_key,
    q1.region_name,
    q1.source_coverage_status,
    q1.project_count AS q1_project_count,
    q1.projects_with_budget AS q1_projects_with_budget,
    q1.budget_coverage_pct AS q1_budget_coverage_pct,
    q1.total_reported_budget_pesos AS q1_budget,
    q1.avg_budget_per_project_pesos AS q1_average,
    COALESCE(gold_regions.project_count, 0) AS gold_project_count,
    COALESCE(distribution.budget_projects, 0) AS budget_projects,
    COALESCE(distribution.zero_budget_projects, 0) AS zero_budget_projects,
    distribution.budget_total,
    ROUND(distribution.median_budget, 2) AS median_budget,
    ROUND(distribution.p90_budget, 2) AS p90_budget,
    distribution.max_budget,
    ROUND(100.0 * distribution.max_budget / NULLIF(distribution.budget_total, 0), 4) AS largest_share_pct
FROM analytics_validation_q1_output AS q1
LEFT JOIN distribution
    ON q1.region_key = distribution.region_key
LEFT JOIN gold_regions
    ON q1.region_key = gold_regions.region_key;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_size_compare AS
SELECT
    COALESCE(expected.region_key, output.region_key) AS region_key,
    COALESCE(output.region_name, expected.region_name) AS region_name,
    COALESCE(output.source_coverage_status, expected.source_coverage_status) AS source_coverage_status,
    output.project_count,
    output.projects_with_budget,
    output.projects_with_zero_budget,
    expected.region_key IS NULL OR output.region_key IS NULL AS region_set_break,
    NOT (output.source_coverage_status <=> expected.source_coverage_status)
    OR NOT (output.project_count <=> expected.q1_project_count)
    OR NOT (output.projects_with_budget <=> expected.q1_projects_with_budget)
    OR NOT (output.budget_coverage_pct <=> expected.q1_budget_coverage_pct)
    OR NOT (output.total_reported_budget_pesos <=> expected.q1_budget)
    OR NOT (output.avg_budget_per_project_pesos <=> expected.q1_average) AS q1_break,
    NOT (output.project_count <=> expected.gold_project_count)
    OR NOT (output.projects_with_budget <=> expected.budget_projects)
    OR NOT (output.total_reported_budget_pesos <=> expected.budget_total) AS gold_break,
    NOT (output.projects_with_zero_budget <=> expected.zero_budget_projects) AS zero_budget_break,
    NOT (output.median_budget_per_project_pesos <=> expected.median_budget)
    OR NOT (output.p90_budget_per_project_pesos <=> expected.p90_budget)
    OR NOT (output.max_budget_per_project_pesos <=> expected.max_budget) AS distribution_break,
    NOT (output.largest_project_share_of_region_budget_pct <=> expected.largest_share_pct) AS largest_share_break,
    output.median_budget_per_project_pesos > output.p90_budget_per_project_pesos
    OR output.p90_budget_per_project_pesos > output.max_budget_per_project_pesos AS order_break,
    COALESCE(output.project_count, 0) = 0
    AND (
        output.median_budget_per_project_pesos IS NOT NULL
        OR output.p90_budget_per_project_pesos IS NOT NULL
        OR output.max_budget_per_project_pesos IS NOT NULL
        OR output.largest_project_share_of_region_budget_pct IS NOT NULL
    ) AS no_project_value_break
FROM analytics_validation_size_expected AS expected
FULL OUTER JOIN analytics_validation_size_output AS output
    ON expected.region_key = output.region_key;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_size_metrics AS
WITH output_metrics AS (
    SELECT
        COUNT(*) AS ps_output_rows,
        COUNT(DISTINCT region_key) AS ps_output_regions,
        COUNT_IF(region_key = 0) AS ps_key0_rows
    FROM analytics_validation_size_output
),

compare_metrics AS (
    SELECT
        COUNT(*) AS ps_compared_regions,
        COUNT_IF(region_set_break) AS ps_region_set_breaks,
        COUNT_IF(q1_break) AS ps_q1_breaks,
        COUNT_IF(gold_break) AS ps_gold_breaks,
        COUNT_IF(zero_budget_break) AS ps_zero_budget_breaks,
        COUNT_IF(distribution_break) AS ps_distribution_breaks,
        COUNT_IF(largest_share_break) AS ps_largest_share_breaks,
        COUNT_IF(order_break) AS ps_order_breaks,
        COUNT_IF(no_project_value_break) AS ps_no_project_value_breaks,
        COALESCE(SUM(projects_with_zero_budget), 0) AS ps_zero_budget_projects,
        COALESCE(SUM(projects_with_budget), 0) AS ps_projects_with_budget,
        COALESCE(SUM(project_count), 0) AS ps_projects,
        COUNT_IF(source_coverage_status <> reported_coverage_status) AS ps_unreported_regions,
        CONCAT_WS('; ', COLLECT_LIST(CASE
            WHEN source_coverage_status <> reported_coverage_status
                THEN CONCAT(region_name, ' ', source_coverage_status)
        END)) AS ps_unreported_detail
    FROM analytics_validation_size_compare
),

lineage_metrics AS (
    SELECT
        COUNT_IF(NOT output.source_snapshot_id <=> context.project_snapshot_id) AS ps_lineage_breaks
    FROM analytics_validation_size_output AS output
    CROSS JOIN analytics_validation_context AS context
)

SELECT *
FROM output_metrics
CROSS JOIN compare_metrics
CROSS JOIN lineage_metrics;

SELECT * FROM analytics_validation_size_compare ORDER BY region_key;

## 13. Implementing office portfolio: recompute from Gold

**What:** rebuild each implementing office from Gold and compare it with `analytics_implementing_office_portfolio`. Then check that the offices add up to every current Gold project and budget.

**Why:**

- Offices are compared with `<=>`, so the `NULL` office row is checked like any other row. It must exist when Gold has projects without a resolved office, and it must have no rank.
- Ranks are rebuilt over real offices only, with `RANK`, so ties share a rank.
- `region_count` and `sector_count` are rebuilt as the size of a distinct set, on their own. Region key `0` and `NULL` sectors are not counted.
- The median is rebuilt from rows with a reported budget only, with the same exact `PERCENTILE_CONT` as the view.

**Expected result:** no break columns set to `true`.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW analytics_validation_office_output AS
SELECT * FROM `05-analytics`.analytics_implementing_office_portfolio;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_office_expected AS
WITH offices AS (
    SELECT
        implementing_office,
        COUNT(*) AS project_count,
        COUNT_IF(reported_budget_pesos IS NOT NULL) AS projects_with_budget,
        SUM(reported_budget_pesos) AS total_reported_budget_pesos,
        SIZE(COLLECT_SET(CASE WHEN region_key <> 0 THEN region_key END)) AS region_count,
        SIZE(COLLECT_SET(standardized_sector)) AS sector_count,
        COUNT_IF(long_running_flag IS NOT NULL) AS long_running_evaluated_count,
        COUNT_IF(long_running_flag) AS long_running_count,
        COUNT_IF(zero_progress_flag IS NOT NULL) AS zero_progress_evaluated_count,
        COUNT_IF(zero_progress_flag) AS zero_progress_count
    FROM analytics_validation_exploratory_current
    GROUP BY ALL
),

office_medians AS (
    -- the median over projects with a reported budget only
    SELECT
        implementing_office,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY reported_budget_pesos) AS median_budget
    FROM analytics_validation_exploratory_current
    WHERE reported_budget_pesos IS NOT NULL
    GROUP BY ALL
),

real_office_ranks AS (
    -- only real offices are ranked. the null office row is left out on purpose
    SELECT
        implementing_office,
        RANK() OVER (ORDER BY project_count DESC) AS project_count_rank,
        RANK() OVER (ORDER BY total_reported_budget_pesos DESC NULLS LAST) AS budget_rank
    FROM offices
    WHERE implementing_office IS NOT NULL
)

SELECT
    offices.*,
    ROUND(offices.total_reported_budget_pesos / NULLIF(offices.projects_with_budget, 0), 2)
        AS avg_budget_per_project_pesos,
    ROUND(office_medians.median_budget, 2) AS median_budget_per_project_pesos,
    real_office_ranks.project_count_rank,
    real_office_ranks.budget_rank
FROM offices
LEFT JOIN office_medians
    ON offices.implementing_office <=> office_medians.implementing_office
LEFT JOIN real_office_ranks
    ON offices.implementing_office = real_office_ranks.implementing_office;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_office_compare AS
SELECT
    COALESCE(expected.implementing_office, output.implementing_office) AS implementing_office,
    -- the office itself can be null, so a missing side is found by its project count
    expected.project_count IS NULL AS extra_in_output,
    output.project_count IS NULL AS missing_from_output,
    NOT (output.project_count <=> expected.project_count)
    OR NOT (output.projects_with_budget <=> expected.projects_with_budget)
    OR NOT (output.total_reported_budget_pesos <=> expected.total_reported_budget_pesos) AS count_or_budget_break,
    NOT (output.avg_budget_per_project_pesos <=> expected.avg_budget_per_project_pesos)
    OR NOT (output.median_budget_per_project_pesos <=> expected.median_budget_per_project_pesos) AS average_break,
    NOT (output.region_count <=> expected.region_count)
    OR NOT (output.sector_count <=> expected.sector_count) AS spread_break,
    NOT (output.long_running_evaluated_count <=> expected.long_running_evaluated_count)
    OR NOT (output.long_running_count <=> expected.long_running_count)
    OR NOT (output.zero_progress_evaluated_count <=> expected.zero_progress_evaluated_count)
    OR NOT (output.zero_progress_count <=> expected.zero_progress_count) AS flag_count_break,
    NOT (output.pct_long_running <=> ROUND(
        100.0 * expected.long_running_count / NULLIF(expected.long_running_evaluated_count, 0), 2
    ))
    OR NOT (output.pct_zero_progress <=> ROUND(
        100.0 * expected.zero_progress_count / NULLIF(expected.zero_progress_evaluated_count, 0), 2
    )) AS rate_break,
    output.long_running_count > output.long_running_evaluated_count
    OR output.long_running_evaluated_count > output.project_count
    OR output.zero_progress_count > output.zero_progress_evaluated_count
    OR output.zero_progress_evaluated_count > output.project_count AS impossible_row,
    NOT (output.project_count_rank <=> expected.project_count_rank)
    OR NOT (output.budget_rank <=> expected.budget_rank) AS rank_break,
    output.project_count IS NOT NULL AND output.implementing_office IS NULL
    AND (output.project_count_rank IS NOT NULL OR output.budget_rank IS NOT NULL) AS null_office_rank_break
FROM analytics_validation_office_expected AS expected
FULL OUTER JOIN analytics_validation_office_output AS output
    ON expected.implementing_office <=> output.implementing_office;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_office_metrics AS
WITH output_metrics AS (
    SELECT
        COUNT(*) AS io_output_rows,
        COUNT(DISTINCT NAMED_STRUCT('office', implementing_office)) AS io_output_keys,
        COALESCE(SUM(project_count), 0) AS io_output_projects,
        COALESCE(SUM(total_reported_budget_pesos), 0) AS io_output_budget
    FROM analytics_validation_office_output
),

gold_metrics AS (
    SELECT
        COUNT(*) AS io_gold_rows,
        COUNT(DISTINCT project_key) AS io_gold_projects,
        COALESCE(SUM(reported_budget_pesos), 0) AS io_gold_budget,
        COUNT_IF(implementing_office IS NULL) AS io_null_office_projects,
        COALESCE(SUM(CASE WHEN implementing_office IS NULL THEN reported_budget_pesos END), 0)
            AS io_null_office_budget,
        COUNT(DISTINCT implementing_office) AS io_real_offices
    FROM analytics_validation_exploratory_current
),

compare_metrics AS (
    SELECT
        COUNT(*) AS io_compared_rows,
        COUNT_IF(extra_in_output OR missing_from_output) AS io_row_set_breaks,
        COUNT_IF(count_or_budget_break) AS io_count_budget_breaks,
        COUNT_IF(average_break) AS io_average_breaks,
        COUNT_IF(spread_break) AS io_spread_breaks,
        COUNT_IF(flag_count_break) AS io_flag_count_breaks,
        COUNT_IF(rate_break) AS io_rate_breaks,
        COUNT_IF(impossible_row) AS io_impossible_rows,
        COUNT_IF(rank_break) AS io_rank_breaks,
        COUNT_IF(null_office_rank_break) AS io_null_office_rank_breaks
    FROM analytics_validation_office_compare
),

lineage_metrics AS (
    SELECT
        COUNT_IF(NOT output.source_snapshot_id <=> context.project_snapshot_id) AS io_lineage_breaks
    FROM analytics_validation_office_output AS output
    CROSS JOIN analytics_validation_context AS context
)

SELECT *
FROM output_metrics
CROSS JOIN gold_metrics
CROSS JOIN compare_metrics
CROSS JOIN lineage_metrics;

SELECT * FROM analytics_validation_office_compare ORDER BY implementing_office NULLS FIRST;

## 14. Turn the metrics into checks

**STOP** checks protect grain, row sets, recomputed measures, ranks, totals, and the coverage and exposure rules. A failed STOP check is `FAIL`.

**FLAG** checks keep known data limits visible with their counts, such as projects on key `0`, missing budgets, unmapped categories, regions with no or limited coverage, projects without an `infra_year` or an implementing office, and zero-progress projects that are For Procurement. A FLAG is not a failure. Zero findings are `PASS`.

Every check names the columns it tested in `details`.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW analytics_validation_q1_metrics AS
WITH current_metrics AS (
    SELECT
        COUNT(*) AS current_rows,
        COUNT(DISTINCT project_key) AS current_projects,
        COALESCE(SUM(reported_budget_pesos), 0) AS current_budget,
        COUNT_IF(region_key = 0) AS key0_projects,
        COALESCE(SUM(CASE WHEN region_key = 0 THEN reported_budget_pesos END), 0) AS key0_budget,
        COUNT_IF(region_key = 0 AND reported_budget_pesos IS NULL) AS key0_missing_budget,
        COUNT_IF(region_key <> 0 AND NOT region_found) AS orphan_projects,
        COALESCE(SUM(CASE WHEN region_key <> 0 AND NOT region_found THEN reported_budget_pesos END), 0) AS orphan_budget,
        COUNT_IF(region_key <> 0 AND region_found) AS included_projects,
        COUNT_IF(region_key <> 0 AND region_found AND reported_budget_pesos IS NULL) AS included_missing_budget,
        COUNT_IF(region_key <> 0 AND region_found AND reported_budget_pesos = 0) AS included_zero_budget,
        COUNT(DISTINCT CASE WHEN region_key <> 0 AND region_found THEN COALESCE(psgc_version, 'NULL') END)
            AS psgc_versions_used
    FROM analytics_validation_q1_current
),

key0_status_counts AS (
    SELECT
        region_match_status,
        COUNT(*) AS projects
    FROM analytics_validation_q1_current
    WHERE region_key = 0
    GROUP BY ALL
),

key0_by_status AS (
    SELECT CONCAT_WS(', ', COLLECT_LIST(CONCAT(region_match_status, '=', projects))) AS key0_detail
    FROM key0_status_counts
),

output_metrics AS (
    SELECT
        COUNT(*) AS output_rows,
        COUNT(DISTINCT region_key) AS output_regions,
        COUNT_IF(region_key = 0) AS output_key0_rows,
        COALESCE(SUM(project_count), 0) AS output_projects,
        COALESCE(SUM(total_reported_budget_pesos), 0) AS output_budget,
        SUM(share_of_included_regions_budget_pct) AS output_share_total
    FROM analytics_validation_q1_output
),

compare_metrics AS (
    SELECT
        COUNT(*) AS compared_regions,
        COUNT_IF(extra_in_output OR missing_from_output) AS region_set_breaks,
        COUNT_IF(project_count_break) AS project_count_breaks,
        COUNT_IF(projects_with_budget_break) AS projects_with_budget_breaks,
        COUNT_IF(budget_break) AS budget_breaks,
        COUNT_IF(coverage_pct_break) AS coverage_pct_breaks,
        COUNT_IF(coordinate_break) AS coordinate_breaks,
        COUNT_IF(average_break) AS average_breaks,
        COUNT_IF(share_break) AS share_breaks,
        COUNT_IF(status_break) AS status_breaks,
        COUNT_IF(rank_break) AS rank_breaks,
        COUNT_IF(rank_gap_break) AS rank_gap_breaks,
        COUNT_IF(budget_without_projects) AS budget_without_project_rows,
        COUNT_IF(is_barmm) AS barmm_regions,
        COUNT_IF(is_barmm AND source_coverage_status = reported_coverage_status) AS barmm_ranked_rows,
        MAX(CASE WHEN is_barmm THEN CONCAT(
            region_name, ' status=', COALESCE(source_coverage_status, 'missing from view'),
            ', projects=', COALESCE(CAST(project_count AS STRING), 'NULL'),
            ', reported budget=', COALESCE(CAST(output_budget_pesos AS STRING), 'NULL')
        ) END) AS barmm_detail,
        COUNT_IF(source_coverage_status <> reported_coverage_status) AS unranked_regions,
        CONCAT_WS('; ', COLLECT_LIST(CASE
            WHEN source_coverage_status <> reported_coverage_status
                THEN CONCAT(region_name, ' ', source_coverage_status)
        END)) AS unranked_detail,
        CONCAT_WS(', ', COLLECT_LIST(CASE
            WHEN extra_in_output OR missing_from_output OR project_count_break OR projects_with_budget_break
                OR budget_break OR coverage_pct_break OR coordinate_break OR average_break OR share_break OR status_break
                OR rank_break OR rank_gap_break OR budget_without_projects
                THEN region_name
        END)) AS break_detail
    FROM analytics_validation_q1_compare
),

lineage_metrics AS (
    SELECT
        COUNT_IF(NOT output.source_snapshot_id <=> context.project_snapshot_id) AS q1_lineage_breaks
    FROM analytics_validation_q1_output AS output
    CROSS JOIN analytics_validation_context AS context
)

SELECT *
FROM current_metrics
CROSS JOIN key0_by_status
CROSS JOIN output_metrics
CROSS JOIN compare_metrics
CROSS JOIN lineage_metrics;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_q2_metrics AS
WITH output_metrics AS (
    SELECT
        COUNT(*) AS q2_output_rows,
        COUNT(DISTINCT NAMED_STRUCT(
            'sector', standardized_sector,
            'flood', is_dpwh_flood_related,
            'status', infra_type_mapping_status
        )) AS q2_output_keys,
        COALESCE(SUM(project_count), 0) AS q2_output_projects,
        COALESCE(SUM(projects_with_budget), 0) AS q2_output_projects_with_budget,
        COALESCE(SUM(total_reported_budget_pesos), 0) AS q2_output_budget,
        SUM(share_of_total_budget_pct) AS q2_share_total
    FROM analytics_validation_q2_output
),

gold_metrics AS (
    SELECT
        COUNT(DISTINCT project_key) AS q2_gold_projects,
        COUNT_IF(reported_budget_pesos IS NOT NULL) AS q2_gold_projects_with_budget,
        COALESCE(SUM(reported_budget_pesos), 0) AS q2_gold_budget
    FROM analytics_validation_q1_current
),

compare_metrics AS (
    SELECT
        COUNT(*) AS q2_compared_rows,
        COUNT_IF(extra_in_output OR missing_from_output) AS q2_row_set_breaks,
        COUNT_IF(count_or_budget_break) AS q2_count_budget_breaks,
        COUNT_IF(average_break) AS q2_average_breaks,
        COUNT_IF(share_break) AS q2_share_breaks,
        COUNT_IF(flood_share_break) AS q2_flood_share_breaks,
        COALESCE(SUM(CASE WHEN is_unmapped THEN expected_projects END), 0) AS q2_unmapped_projects,
        COALESCE(SUM(CASE WHEN is_unmapped THEN expected_budget END), 0) AS q2_unmapped_budget,
        COALESCE(SUM(CASE WHEN is_multi_sector_flood THEN expected_projects END), 0) AS q2_multi_flood_projects,
        COALESCE(SUM(CASE WHEN is_multi_sector_flood THEN expected_budget END), 0) AS q2_multi_flood_budget
    FROM analytics_validation_q2_compare
),

lineage_metrics AS (
    SELECT
        COUNT_IF(NOT output.source_snapshot_id <=> context.project_snapshot_id) AS q2_lineage_breaks
    FROM analytics_validation_q2_output AS output
    CROSS JOIN analytics_validation_context AS context
)

SELECT *
FROM output_metrics
CROSS JOIN gold_metrics
CROSS JOIN compare_metrics
CROSS JOIN lineage_metrics;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_q3_metrics AS
WITH output_metrics AS (
    SELECT
        COUNT(*) AS q3_output_rows,
        COUNT(DISTINCT NAMED_STRUCT(
            'region', region_key,
            'sector', standardized_sector,
            'sector_reason', infra_type_mapping_status,
            'status', status_group,
            'status_reason', status_mapping_status
        )) AS q3_output_keys,
        COUNT_IF(
            long_running_count > long_running_evaluated_count
            OR long_running_evaluated_count > project_count
            OR zero_progress_count > zero_progress_evaluated_count
            OR zero_progress_evaluated_count > project_count
        ) AS q3_impossible_rows
    FROM analytics_validation_q3_output
),

group_metrics AS (
    SELECT
        COUNT(*) AS q3_compared_groups,
        COUNT_IF(extra_in_output OR missing_from_output) AS q3_group_set_breaks,
        COUNT_IF(count_break) AS q3_count_breaks,
        COUNT_IF(rate_break) AS q3_rate_breaks,
        COALESCE(SUM(CASE WHEN is_unknown_status THEN expected_projects END), 0) AS q3_unknown_status_projects,
        COALESCE(SUM(expected_unevaluated_long_running), 0) AS q3_unevaluated_long_running,
        COALESCE(SUM(expected_projects), 0) AS q3_expected_projects
    FROM analytics_validation_q3_compare
),

share_metrics AS (
    SELECT
        COUNT_IF(share_break OR sector_share_break) AS q3_share_breaks,
        COUNT_IF(status_break) AS q3_status_breaks
    FROM analytics_validation_q3_share_compare
),

region_metrics AS (
    SELECT
        COUNT(*) AS q3_compared_regions,
        COUNT_IF(region_set_break OR project_total_break) AS q3_region_breaks,
        COUNT_IF(empty_row_break) AS q3_empty_row_breaks
    FROM analytics_validation_q3_region_compare
),

lineage_metrics AS (
    SELECT
        COUNT_IF(NOT output.source_snapshot_id <=> context.project_snapshot_id) AS q3_lineage_breaks
    FROM analytics_validation_q3_output AS output
    CROSS JOIN analytics_validation_context AS context
)

SELECT *
FROM output_metrics
CROSS JOIN group_metrics
CROSS JOIN share_metrics
CROSS JOIN region_metrics
CROSS JOIN lineage_metrics;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_q4_metrics AS
WITH output_metrics AS (
    SELECT
        COUNT(*) AS q4_output_rows,
        COUNT(DISTINCT region_key) AS q4_output_regions,
        COUNT_IF(region_key = 0) AS q4_key0_rows,
        COUNT(DISTINCT population_reference_year) AS q4_reference_years
    FROM analytics_validation_q4_output
),

compare_metrics AS (
    SELECT
        COUNT(*) AS q4_compared_regions,
        COUNT_IF(region_set_break) AS q4_region_set_breaks,
        COUNT_IF(q1_break) AS q4_q1_breaks,
        COUNT_IF(population_break) AS q4_population_breaks,
        COUNT_IF(share_break) AS q4_share_breaks,
        COUNT_IF(gap_break) AS q4_gap_breaks,
        COUNT_IF(per_capita_break) AS q4_per_capita_breaks,
        COUNT_IF(unreported_ratio_break) AS q4_unreported_ratio_breaks,
        COUNT_IF(missing_population) AS q4_missing_population,
        SUM(investment_share_pct) AS q4_investment_share_total,
        SUM(population_share_pct) AS q4_population_share_total,
        SUM(comparable_population_share_pct) AS q4_comparable_share_total,
        COUNT_IF(not_comparable) AS q4_unreported_regions,
        CONCAT_WS('; ', COLLECT_LIST(CASE
            WHEN not_comparable THEN CONCAT(region_name, ' ', source_coverage_status)
        END)) AS q4_unreported_detail
    FROM analytics_validation_q4_compare
),

lineage_metrics AS (
    SELECT
        COUNT_IF(NOT output.source_snapshot_id <=> context.project_snapshot_id) AS q4_lineage_breaks
    FROM analytics_validation_q4_output AS output
    CROSS JOIN analytics_validation_context AS context
)

SELECT *
FROM output_metrics
CROSS JOIN compare_metrics
CROSS JOIN lineage_metrics;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_q5_metrics AS
WITH output_metrics AS (
    SELECT
        COUNT(*) AS q5_output_rows,
        COUNT(DISTINCT region_key) AS q5_output_regions
    FROM analytics_validation_q5_output
),

compare_metrics AS (
    SELECT
        COUNT(*) AS q5_compared_regions,
        COUNT_IF(region_set_break) AS q5_region_set_breaks,
        COUNT_IF(q1_break) AS q5_q1_breaks,
        COUNT_IF(flood_break) AS q5_flood_breaks,
        COUNT_IF(flood_share_break) AS q5_flood_share_breaks,
        COUNT_IF(exposure_status_break) AS q5_exposure_status_breaks,
        COUNT_IF(invalid_status) AS q5_invalid_statuses,
        COUNT_IF(level_area_break) AS q5_level_area_breaks,
        COUNT_IF(non_additive_total_break) AS q5_non_additive_totals,
        COUNT_IF(total_area_break) AS q5_total_area_breaks,
        COUNT_IF(pesos_per_sqkm_break) AS q5_pesos_per_sqkm_breaks,
        COUNT_IF(unknown_shown_as_value_break) AS q5_unknown_shown_as_value,
        COUNT_IF(point_break) AS q5_point_breaks,
        COUNT_IF(point_sum_break) AS q5_point_sum_breaks,
        COALESCE(SUM(flood_control_project_count), 0) AS q5_flood_control_projects,
        COALESCE(SUM(flood_control_points_outside_mapped_zones), 0) AS q5_points_outside,
        COALESCE(SUM(flood_control_points_no_usable_point), 0) AS q5_points_no_usable,
        COUNT_IF(exposure_area_status = 'NON_ADDITIVE_CROSS_LEVEL_OVERLAP') AS q5_non_additive_regions,
        CONCAT_WS(', ', COLLECT_LIST(CASE
            WHEN exposure_area_status = 'NON_ADDITIVE_CROSS_LEVEL_OVERLAP' THEN region_name
        END)) AS q5_non_additive_detail,
        COUNT_IF(exposure_area_status IN ('NOT_EVALUATED', 'NO_EXPOSURE_DATA')) AS q5_no_exposure_regions,
        CONCAT_WS('; ', COLLECT_LIST(CASE
            WHEN exposure_area_status IN ('NOT_EVALUATED', 'NO_EXPOSURE_DATA')
                THEN CONCAT(region_name, ' ', exposure_area_status)
        END)) AS q5_no_exposure_detail
    FROM analytics_validation_q5_compare
),

lineage_metrics AS (
    SELECT
        COUNT_IF(NOT output.source_snapshot_id <=> context.project_snapshot_id
            OR NOT output.mgb_source_snapshot_id <=> context.mgb_source_snapshot_id) AS q5_lineage_breaks
    FROM analytics_validation_q5_output AS output
    CROSS JOIN analytics_validation_context AS context
)

SELECT *
FROM output_metrics
CROSS JOIN compare_metrics
CROSS JOIN lineage_metrics;

CREATE OR REPLACE TEMPORARY VIEW analytics_validation_q6_metrics AS
WITH output_metrics AS (
    SELECT
        COUNT(*) AS q6_output_rows,
        COUNT(DISTINCT NAMED_STRUCT(
            'region', region_key, 'sector', standardized_sector, 'sector_reason', infra_type_mapping_status
        )) AS q6_output_keys
    FROM analytics_validation_q6_output
),

group_metrics AS (
    SELECT
        COUNT(*) AS q6_compared_groups,
        COUNT_IF(extra_in_output OR missing_from_output) AS q6_group_set_breaks,
        COUNT_IF(count_or_budget_break) AS q6_count_budget_breaks,
        COUNT_IF(average_break) AS q6_average_breaks,
        COUNT_IF(share_break) AS q6_share_breaks
    FROM analytics_validation_q6_compare
),

region_metrics AS (
    SELECT
        COUNT(*) AS q6_compared_regions,
        COUNT_IF(region_set_break OR q1_break) AS q6_region_breaks,
        COUNT_IF(empty_row_break) AS q6_empty_row_breaks
    FROM analytics_validation_q6_region_compare
),

lineage_metrics AS (
    SELECT
        COUNT_IF(NOT output.source_snapshot_id <=> context.project_snapshot_id) AS q6_lineage_breaks
    FROM analytics_validation_q6_output AS output
    CROSS JOIN analytics_validation_context AS context
)

SELECT *
FROM output_metrics
CROSS JOIN group_metrics
CROSS JOIN region_metrics
CROSS JOIN lineage_metrics;


CREATE OR REPLACE TEMPORARY VIEW analytics_validation_checks AS
WITH q1_checks AS (
    SELECT INLINE(ARRAY(
        NAMED_STRUCT(
            'table_name', 'analytics_regional_investment', 'attribute', 'Uniqueness',
            'check_name', 'At most one current row per project',
            'failed_rows', CAST(m.current_rows - m.current_projects AS BIGINT), 'total_rows', CAST(m.current_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: fact_project_snapshot.project_key, is_current_snapshot. Current rows=', m.current_rows)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_regional_investment', 'attribute', 'Uniqueness',
            'check_name', 'One row per region',
            'failed_rows', CAST(m.output_rows - m.output_regions AS BIGINT), 'total_rows', CAST(m.output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_key'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_regional_investment', 'attribute', 'Completeness',
            'check_name', 'Every official region of the current PSGC version has exactly one row',
            'failed_rows', CAST(m.region_set_breaks AS BIGINT), 'total_rows', CAST(m.compared_regions AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: region_key, dim_region.psgc_version. Regions with a break: ', COALESCE(NULLIF(m.break_detail, ''), 'none'))
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_regional_investment', 'attribute', 'Consistency',
            'check_name', 'Current projects use one PSGC version',
            'failed_rows', CAST(GREATEST(m.psgc_versions_used - 1, 0) AS BIGINT), 'total_rows', CAST(m.psgc_versions_used AS BIGINT),
            'action', 'stop',
            'details', 'Columns: dim_region.psgc_version'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_regional_investment', 'attribute', 'Validity',
            'check_name', 'Region key 0 is never in the view',
            'failed_rows', CAST(m.output_key0_rows AS BIGINT), 'total_rows', CAST(m.output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_key'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_regional_investment', 'attribute', 'Consistency',
            'check_name', 'Every region key above 0 finds its dim_region row',
            'failed_rows', CAST(m.orphan_projects AS BIGINT), 'total_rows', CAST(m.current_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: fact_project_snapshot.region_key. Reported budget on unknown keys=', m.orphan_budget)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_regional_investment', 'attribute', 'Accuracy',
            'check_name', 'Project count matches Gold for every region',
            'failed_rows', CAST(m.project_count_breaks AS BIGINT), 'total_rows', CAST(m.compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: project_count'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_regional_investment', 'attribute', 'Accuracy',
            'check_name', 'Projects with a budget match Gold for every region',
            'failed_rows', CAST(m.projects_with_budget_breaks AS BIGINT), 'total_rows', CAST(m.compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: projects_with_budget'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_regional_investment', 'attribute', 'Accuracy',
            'check_name', 'Reported budget matches Gold for every region and a NULL budget stays NULL',
            'failed_rows', CAST(m.budget_breaks AS BIGINT), 'total_rows', CAST(m.compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: total_reported_budget_pesos. Compared with a NULL-safe equal'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_regional_investment', 'attribute', 'Accuracy',
            'check_name', 'Budget coverage percent is projects with a budget over all projects',
            'failed_rows', CAST(m.coverage_pct_breaks AS BIGINT), 'total_rows', CAST(m.compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: budget_coverage_pct, projects_with_budget, project_count'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_regional_investment', 'attribute', 'Accuracy',
            'check_name', 'Coordinate and match coverage match Gold for every region',
            'failed_rows', CAST(m.coordinate_breaks AS BIGINT), 'total_rows', CAST(m.compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: projects_with_coordinates, coordinate_coverage_pct, projects_matched_with_conflict'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_regional_investment', 'attribute', 'Accuracy',
            'check_name', 'Average budget divides by projects with a budget',
            'failed_rows', CAST(m.average_breaks AS BIGINT), 'total_rows', CAST(m.compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: avg_budget_per_project_pesos, projects_with_budget'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_regional_investment', 'attribute', 'Accuracy',
            'check_name', 'Budget share matches Gold for every region',
            'failed_rows', CAST(m.share_breaks AS BIGINT), 'total_rows', CAST(m.compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: share_of_included_regions_budget_pct'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_regional_investment', 'attribute', 'Accuracy',
            'check_name', 'Budget shares add up to 100 percent',
            'failed_rows', CAST(CASE WHEN m.output_budget > 0 AND NOT ABS(m.output_share_total - 100) <= share_total_tolerance_pct THEN 1 ELSE 0 END AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: share_of_included_regions_budget_pct. Total=', COALESCE(CAST(m.output_share_total AS STRING), 'NULL'), ', tolerance=', share_total_tolerance_pct)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_regional_investment', 'attribute', 'Consistency',
            'check_name', 'Coverage status follows the rule',
            'failed_rows', CAST(m.status_breaks AS BIGINT), 'total_rows', CAST(m.compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: source_coverage_status. No mapped project is NO_MAPPED_PROJECT_DATA, BARMM with projects is LIMITED_SOURCE_COVERAGE, the rest is REPORTED'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_regional_investment', 'attribute', 'Accuracy',
            'check_name', 'Budget and project-count ranks cover only REPORTED regions and rank 1 is the highest',
            'failed_rows', CAST(m.rank_breaks AS BIGINT), 'total_rows', CAST(m.compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: budget_rank, project_count_rank'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_regional_investment', 'attribute', 'Accuracy',
            'check_name', 'Rank gap is project-count rank minus budget rank',
            'failed_rows', CAST(m.rank_gap_breaks AS BIGINT), 'total_rows', CAST(m.compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: project_vs_budget_rank_gap'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_regional_investment', 'attribute', 'Validity',
            'check_name', 'A region with no mapped project shows no budget',
            'failed_rows', CAST(m.budget_without_project_rows AS BIGINT), 'total_rows', CAST(m.compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: project_count, total_reported_budget_pesos'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_regional_investment', 'attribute', 'Validity',
            'check_name', 'BARMM is never ranked',
            'failed_rows', CAST(m.barmm_ranked_rows AS BIGINT), 'total_rows', CAST(m.barmm_regions AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: dim_region.is_barmm, source_coverage_status. ', COALESCE(m.barmm_detail, 'No BARMM row in dim_region'))
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_regional_investment', 'attribute', 'Consistency',
            'check_name', 'View, key 0, and unknown keys add up to every current project and budget',
            'failed_rows', CAST(CASE WHEN m.output_projects + m.key0_projects + m.orphan_projects <> m.current_projects OR m.output_budget + m.key0_budget + m.orphan_budget <> m.current_budget THEN 1 ELSE 0 END AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: project_count, total_reported_budget_pesos. View projects=', m.output_projects, ', key 0=', m.key0_projects, ', unknown keys=', m.orphan_projects, ', all current=', m.current_projects, '. View budget=', m.output_budget, ', key 0=', m.key0_budget, ', all current=', m.current_budget)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_regional_investment', 'attribute', 'Completeness',
            'check_name', 'Current projects left out on region key 0',
            'failed_rows', CAST(m.key0_projects AS BIGINT), 'total_rows', CAST(m.current_rows AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Columns: region_key, region_match_status. By status: ', COALESCE(NULLIF(m.key0_detail, ''), 'none'), '. Reported budget=', m.key0_budget, ', without a budget=', m.key0_missing_budget)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_regional_investment', 'attribute', 'Completeness',
            'check_name', 'Projects in the view without a reported budget',
            'failed_rows', CAST(m.included_missing_budget AS BIGINT), 'total_rows', CAST(m.included_projects AS BIGINT),
            'action', 'flag',
            'details', 'Columns: reported_budget_pesos. They count in project_count but not in projects_with_budget or the average'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_regional_investment', 'attribute', 'Completeness',
            'check_name', 'Projects in the view with a reported budget of 0',
            'failed_rows', CAST(m.included_zero_budget AS BIGINT), 'total_rows', CAST(m.included_projects AS BIGINT),
            'action', 'flag',
            'details', 'Columns: reported_budget_pesos. A 0 stays 0 as reported and lowers the average'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_regional_investment', 'attribute', 'Completeness',
            'check_name', 'Official regions with no or limited DPWH project coverage',
            'failed_rows', CAST(m.unranked_regions AS BIGINT), 'total_rows', CAST(m.compared_regions AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Columns: source_coverage_status. Not ranked: ', COALESCE(NULLIF(m.unranked_detail, ''), 'none'))
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_regional_investment', 'attribute', 'Auditability',
            'check_name', 'Snapshot ID is the current DPWH snapshot',
            'failed_rows', CAST(m.q1_lineage_breaks AS BIGINT), 'total_rows', CAST(m.output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: source_snapshot_id (D-29)'
        )
    ))
    FROM analytics_validation_q1_metrics AS m
),

q2_checks AS (
    SELECT INLINE(ARRAY(
        NAMED_STRUCT(
            'table_name', 'analytics_category_portfolio', 'attribute', 'Uniqueness',
            'check_name', 'One row per sector, flood flag, and mapping status',
            'failed_rows', CAST(m.q2_output_rows - m.q2_output_keys AS BIGINT), 'total_rows', CAST(m.q2_output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: standardized_sector, is_dpwh_flood_related, infra_type_mapping_status'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_category_portfolio', 'attribute', 'Completeness',
            'check_name', 'Every Gold category combination has exactly one row',
            'failed_rows', CAST(m.q2_row_set_breaks AS BIGINT), 'total_rows', CAST(m.q2_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: standardized_sector, is_dpwh_flood_related, infra_type_mapping_status'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_category_portfolio', 'attribute', 'Accuracy',
            'check_name', 'Project count, projects with a budget, and budget match Gold for every category',
            'failed_rows', CAST(m.q2_count_budget_breaks AS BIGINT), 'total_rows', CAST(m.q2_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: project_count, projects_with_budget, total_reported_budget_pesos. Compared with a NULL-safe equal'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_category_portfolio', 'attribute', 'Accuracy',
            'check_name', 'Average budget divides by projects with a budget',
            'failed_rows', CAST(m.q2_average_breaks AS BIGINT), 'total_rows', CAST(m.q2_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: avg_budget_per_project_pesos, projects_with_budget'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_category_portfolio', 'attribute', 'Accuracy',
            'check_name', 'Budget share matches Gold for every category',
            'failed_rows', CAST(m.q2_share_breaks AS BIGINT), 'total_rows', CAST(m.q2_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: share_of_total_budget_pct'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_category_portfolio', 'attribute', 'Accuracy',
            'check_name', 'Broad and strict flood control shares match Gold',
            'failed_rows', CAST(m.q2_flood_share_breaks AS BIGINT), 'total_rows', CAST(m.q2_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: flood_control_share_of_portfolio_pct (is_dpwh_flood_related), flood_control_single_sector_share_pct (standardized_sector)'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_category_portfolio', 'attribute', 'Accuracy',
            'check_name', 'Budget shares add up to 100 percent',
            'failed_rows', CAST(CASE WHEN m.q2_output_budget > 0 AND NOT ABS(m.q2_share_total - 100) <= share_total_tolerance_pct THEN 1 ELSE 0 END AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: share_of_total_budget_pct. Total=', COALESCE(CAST(m.q2_share_total AS STRING), 'NULL'), ', tolerance=', share_total_tolerance_pct)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_category_portfolio', 'attribute', 'Consistency',
            'check_name', 'Categories add up to every current Gold project and budget, including region key 0',
            'failed_rows', CAST(CASE WHEN m.q2_output_projects <> m.q2_gold_projects OR m.q2_output_projects_with_budget <> m.q2_gold_projects_with_budget OR m.q2_output_budget <> m.q2_gold_budget THEN 1 ELSE 0 END AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: project_count, total_reported_budget_pesos. View projects=', m.q2_output_projects, ', Gold=', m.q2_gold_projects, '. View budget=', m.q2_output_budget, ', Gold=', m.q2_gold_budget)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_category_portfolio', 'attribute', 'Completeness',
            'check_name', 'Projects with no usable category (sector NULL)',
            'failed_rows', CAST(m.q2_unmapped_projects AS BIGINT), 'total_rows', CAST(m.q2_gold_projects AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Columns: standardized_sector, infra_type_mapping_status. Reported budget=', m.q2_unmapped_budget)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_category_portfolio', 'attribute', 'Consistency',
            'check_name', 'Multi-sector projects counted whole in the flood control share',
            'failed_rows', CAST(m.q2_multi_flood_projects AS BIGINT), 'total_rows', CAST(m.q2_gold_projects AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Columns: standardized_sector, is_dpwh_flood_related. D-39 flags a project with any Flood Control and Drainage component. Reported budget of these projects=', m.q2_multi_flood_budget)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_category_portfolio', 'attribute', 'Auditability',
            'check_name', 'Snapshot ID is the current DPWH snapshot',
            'failed_rows', CAST(m.q2_lineage_breaks AS BIGINT), 'total_rows', CAST(m.q2_output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: source_snapshot_id (D-29)'
        )
    ))
    FROM analytics_validation_q2_metrics AS m
),

q3_checks AS (
    SELECT INLINE(ARRAY(
        NAMED_STRUCT(
            'table_name', 'analytics_project_delivery_status', 'attribute', 'Uniqueness',
            'check_name', 'One row per region, category, and status group',
            'failed_rows', CAST(m.q3_output_rows - m.q3_output_keys AS BIGINT), 'total_rows', CAST(m.q3_output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_key, standardized_sector, status_group'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_project_delivery_status', 'attribute', 'Completeness',
            'check_name', 'Every Gold region, category, and status group has exactly one row',
            'failed_rows', CAST(m.q3_group_set_breaks AS BIGINT), 'total_rows', CAST(m.q3_compared_groups AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_key, standardized_sector, status_group'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_project_delivery_status', 'attribute', 'Accuracy',
            'check_name', 'Project and flag counts match Gold for every group',
            'failed_rows', CAST(m.q3_count_breaks AS BIGINT), 'total_rows', CAST(m.q3_compared_groups AS BIGINT),
            'action', 'stop',
            'details', 'Columns: project_count, long_running_evaluated_count, long_running_count, zero_progress_evaluated_count, zero_progress_count'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_project_delivery_status', 'attribute', 'Accuracy',
            'check_name', 'Long-running and zero-progress rates divide by projects where the flag is known',
            'failed_rows', CAST(m.q3_rate_breaks AS BIGINT), 'total_rows', CAST(m.q3_compared_groups AS BIGINT),
            'action', 'stop',
            'details', 'Columns: pct_long_running, pct_zero_progress'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_project_delivery_status', 'attribute', 'Validity',
            'check_name', 'Flag counts never exceed their denominators',
            'failed_rows', CAST(m.q3_impossible_rows AS BIGINT), 'total_rows', CAST(m.q3_output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: project_count, long_running_evaluated_count, long_running_count, zero_progress_evaluated_count, zero_progress_count'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_project_delivery_status', 'attribute', 'Accuracy',
            'check_name', 'Status shares divide by the region and the region-sector project counts',
            'failed_rows', CAST(m.q3_share_breaks AS BIGINT), 'total_rows', CAST(m.q3_output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: pct_of_region_projects, pct_of_region_sector_projects, analytics_regional_investment.project_count'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_project_delivery_status', 'attribute', 'Consistency',
            'check_name', 'Regions, project totals, and coverage status match Q1',
            'failed_rows', CAST(m.q3_region_breaks + m.q3_status_breaks AS BIGINT), 'total_rows', CAST(m.q3_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_key, project_count, source_coverage_status'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_project_delivery_status', 'attribute', 'Completeness',
            'check_name', 'A region with no mapped project keeps exactly one empty row',
            'failed_rows', CAST(m.q3_empty_row_breaks AS BIGINT), 'total_rows', CAST(m.q3_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: standardized_sector, status_group, project_count'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_project_delivery_status', 'attribute', 'Completeness',
            'check_name', 'Projects with no mapped status group (status_group NULL)',
            'failed_rows', CAST(m.q3_unknown_status_projects AS BIGINT), 'total_rows', CAST(m.q3_expected_projects AS BIGINT),
            'action', 'flag',
            'details', 'Columns: status_group, status_mapping_status. NO_SOURCE_STATUS means Gold has no status; NO_APPROVED_STATUS_RULE means the source status, often a shifted value, has no approved mapping'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_project_delivery_status', 'attribute', 'Completeness',
            'check_name', 'Projects without a long-running flag because they have no start date',
            'failed_rows', CAST(m.q3_unevaluated_long_running AS BIGINT), 'total_rows', CAST(m.q3_expected_projects AS BIGINT),
            'action', 'flag',
            'details', 'Columns: long_running_evaluated_count. These are left out of pct_long_running'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_project_delivery_status', 'attribute', 'Auditability',
            'check_name', 'Snapshot ID is the current DPWH snapshot',
            'failed_rows', CAST(m.q3_lineage_breaks AS BIGINT), 'total_rows', CAST(m.q3_output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: source_snapshot_id (D-29)'
        )
    ))
    FROM analytics_validation_q3_metrics AS m
),

q4_checks AS (
    SELECT INLINE(ARRAY(
        NAMED_STRUCT(
            'table_name', 'analytics_population_equity', 'attribute', 'Uniqueness',
            'check_name', 'One row per region',
            'failed_rows', CAST(m.q4_output_rows - m.q4_output_regions AS BIGINT), 'total_rows', CAST(m.q4_output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_key'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_population_equity', 'attribute', 'Completeness',
            'check_name', 'Every Q1 region has exactly one row',
            'failed_rows', CAST(m.q4_region_set_breaks AS BIGINT), 'total_rows', CAST(m.q4_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_key'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_population_equity', 'attribute', 'Consistency',
            'check_name', 'Budget and coverage status match Q1',
            'failed_rows', CAST(m.q4_q1_breaks AS BIGINT), 'total_rows', CAST(m.q4_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: total_reported_budget_pesos, source_coverage_status'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_population_equity', 'attribute', 'Accuracy',
            'check_name', 'Population matches the newest primary Gold population',
            'failed_rows', CAST(m.q4_population_breaks AS BIGINT), 'total_rows', CAST(m.q4_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: population_count, population_reference_year'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_population_equity', 'attribute', 'Consistency',
            'check_name', 'One population reference year',
            'failed_rows', CAST(GREATEST(m.q4_reference_years - 1, 0) AS BIGINT), 'total_rows', CAST(m.q4_reference_years AS BIGINT),
            'action', 'stop',
            'details', 'Columns: population_reference_year'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_population_equity', 'attribute', 'Accuracy',
            'check_name', 'Investment, population, and comparable population shares match Gold',
            'failed_rows', CAST(m.q4_share_breaks AS BIGINT), 'total_rows', CAST(m.q4_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: investment_share_pct, population_share_pct, comparable_population_share_pct'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_population_equity', 'attribute', 'Accuracy',
            'check_name', 'Each share adds up to 100 percent',
            'failed_rows', CAST((CASE WHEN m.q4_investment_share_total IS NOT NULL AND NOT ABS(m.q4_investment_share_total - 100) <= share_total_tolerance_pct THEN 1 ELSE 0 END) + (CASE WHEN m.q4_population_share_total IS NOT NULL AND NOT ABS(m.q4_population_share_total - 100) <= share_total_tolerance_pct THEN 1 ELSE 0 END) + (CASE WHEN m.q4_comparable_share_total IS NOT NULL AND NOT ABS(m.q4_comparable_share_total - 100) <= share_total_tolerance_pct THEN 1 ELSE 0 END) AS BIGINT), 'total_rows', CAST(3 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: investment_share_pct, population_share_pct, comparable_population_share_pct. Totals=', COALESCE(CAST(m.q4_investment_share_total AS STRING), 'NULL'), ', ', COALESCE(CAST(m.q4_population_share_total AS STRING), 'NULL'), ', ', COALESCE(CAST(m.q4_comparable_share_total AS STRING), 'NULL'))
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_population_equity', 'attribute', 'Accuracy',
            'check_name', 'Equity gap is investment share minus comparable population share',
            'failed_rows', CAST(m.q4_gap_breaks AS BIGINT), 'total_rows', CAST(m.q4_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: equity_gap_pp'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_population_equity', 'attribute', 'Accuracy',
            'check_name', 'Investment per capita divides comparable budget by comparable population',
            'failed_rows', CAST(m.q4_per_capita_breaks AS BIGINT), 'total_rows', CAST(m.q4_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: investment_per_capita_pesos'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_population_equity', 'attribute', 'Validity',
            'check_name', 'A region outside the comparable set shows no share, gap, or per-capita value',
            'failed_rows', CAST(m.q4_unreported_ratio_breaks AS BIGINT), 'total_rows', CAST(m.q4_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: investment_share_pct, comparable_population_share_pct, equity_gap_pp, investment_per_capita_pesos'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_population_equity', 'attribute', 'Validity',
            'check_name', 'Per-person ratios leave region key 0 out',
            'failed_rows', CAST(m.q4_key0_rows AS BIGINT), 'total_rows', CAST(m.q4_output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_key. Asked for by the Gold validation docs once the dashboard datasets exist'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_population_equity', 'attribute', 'Completeness',
            'check_name', 'Official regions without a primary population',
            'failed_rows', CAST(m.q4_missing_population AS BIGINT), 'total_rows', CAST(m.q4_compared_regions AS BIGINT),
            'action', 'flag',
            'details', 'Columns: population_count'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_population_equity', 'attribute', 'Completeness',
            'check_name', 'Regions left out of the equity comparison',
            'failed_rows', CAST(m.q4_unreported_regions AS BIGINT), 'total_rows', CAST(m.q4_compared_regions AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Columns: source_coverage_status, total_reported_budget_pesos, population_count. Left out: ', COALESCE(NULLIF(m.q4_unreported_detail, ''), 'none'))
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_population_equity', 'attribute', 'Auditability',
            'check_name', 'Snapshot ID is the current DPWH snapshot',
            'failed_rows', CAST(m.q4_lineage_breaks AS BIGINT), 'total_rows', CAST(m.q4_output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: source_snapshot_id (D-29)'
        )
    ))
    FROM analytics_validation_q4_metrics AS m
),

q5_checks AS (
    SELECT INLINE(ARRAY(
        NAMED_STRUCT(
            'table_name', 'analytics_flood_control_exposure', 'attribute', 'Uniqueness',
            'check_name', 'One row per region',
            'failed_rows', CAST(m.q5_output_rows - m.q5_output_regions AS BIGINT), 'total_rows', CAST(m.q5_output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_key'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_control_exposure', 'attribute', 'Completeness',
            'check_name', 'Every Q1 region has exactly one row',
            'failed_rows', CAST(m.q5_region_set_breaks AS BIGINT), 'total_rows', CAST(m.q5_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_key'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_control_exposure', 'attribute', 'Consistency',
            'check_name', 'Budget and coverage status match Q1',
            'failed_rows', CAST(m.q5_q1_breaks AS BIGINT), 'total_rows', CAST(m.q5_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: total_reported_budget_pesos, source_coverage_status'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_control_exposure', 'attribute', 'Accuracy',
            'check_name', 'Flood control projects and budget match Gold for every region',
            'failed_rows', CAST(m.q5_flood_breaks AS BIGINT), 'total_rows', CAST(m.q5_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: flood_control_project_count, flood_control_budget_pesos'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_control_exposure', 'attribute', 'Accuracy',
            'check_name', 'Flood control share of the region budget matches Gold',
            'failed_rows', CAST(m.q5_flood_share_breaks AS BIGINT), 'total_rows', CAST(m.q5_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: flood_control_share_of_region_budget_pct'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_control_exposure', 'attribute', 'Validity',
            'check_name', 'Exposure status is a known value',
            'failed_rows', CAST(m.q5_invalid_statuses AS BIGINT), 'total_rows', CAST(m.q5_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: exposure_area_status'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_control_exposure', 'attribute', 'Accuracy',
            'check_name', 'Exposure status matches the Gold additivity status',
            'failed_rows', CAST(m.q5_exposure_status_breaks AS BIGINT), 'total_rows', CAST(m.q5_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: exposure_area_status, fact_region_flood_exposure.level_area_additivity_status'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_control_exposure', 'attribute', 'Accuracy',
            'check_name', 'Each level area and share matches the newest Gold exposure run',
            'failed_rows', CAST(m.q5_level_area_breaks AS BIGINT), 'total_rows', CAST(m.q5_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: low_area_sqkm, moderate_area_sqkm, high_area_sqkm, very_high_area_sqkm, high_share_of_region_area_pct, very_high_share_of_region_area_pct'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_control_exposure', 'attribute', 'Validity',
            'check_name', 'A total area appears only when the levels may be added',
            'failed_rows', CAST(m.q5_non_additive_totals AS BIGINT), 'total_rows', CAST(m.q5_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: total_susceptible_area_sqkm, exposure_area_status'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_control_exposure', 'attribute', 'Accuracy',
            'check_name', 'Total area is the sum of the level areas where they may be added',
            'failed_rows', CAST(m.q5_total_area_breaks AS BIGINT), 'total_rows', CAST(m.q5_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: total_susceptible_area_sqkm'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_control_exposure', 'attribute', 'Accuracy',
            'check_name', 'Pesos per square kilometer divides flood control budget by the total area',
            'failed_rows', CAST(m.q5_pesos_per_sqkm_breaks AS BIGINT), 'total_rows', CAST(m.q5_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: flood_control_pesos_per_sqkm_exposed'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_control_exposure', 'attribute', 'Validity',
            'check_name', 'A region with no mapped DPWH project shows no flood control value',
            'failed_rows', CAST(m.q5_unknown_shown_as_value AS BIGINT), 'total_rows', CAST(m.q5_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: flood_control_project_count, flood_control_share_of_region_budget_pct, flood_control_pesos_per_sqkm_exposed, flood_control_points_*'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_control_exposure', 'attribute', 'Accuracy',
            'check_name', 'Project-point groups and their percents match Gold',
            'failed_rows', CAST(m.q5_point_breaks AS BIGINT), 'total_rows', CAST(m.q5_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: flood_control_points_*'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_control_exposure', 'attribute', 'Consistency',
            'check_name', 'The five project-point groups add up to the flood control project count',
            'failed_rows', CAST(m.q5_point_sum_breaks AS BIGINT), 'total_rows', CAST(m.q5_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: flood_control_points_*, flood_control_project_count'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_control_exposure', 'attribute', 'Completeness',
            'check_name', 'Flood control projects not placed in any MGB level by their map point',
            'failed_rows', CAST(m.q5_points_outside + m.q5_points_no_usable AS BIGINT),
            'total_rows', CAST(m.q5_flood_control_projects AS BIGINT),
            'action', 'flag',
            'details', CONCAT(
                'Columns: flood_control_points_outside_mapped_zones, flood_control_points_no_usable_point. Outside every mapped zone, not low risk: ', CAST(m.q5_points_outside AS STRING),
                '. No usable point: ', CAST(m.q5_points_no_usable AS STRING), '. Ambiguous projects are counted separately.'
            )
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_control_exposure', 'attribute', 'Completeness',
            'check_name', 'Regions whose flood levels overlap, so no total area',
            'failed_rows', CAST(m.q5_non_additive_regions AS BIGINT), 'total_rows', CAST(m.q5_compared_regions AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Columns: exposure_area_status. Per-level areas are still shown. Regions: ', COALESCE(NULLIF(m.q5_non_additive_detail, ''), 'none'))
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_control_exposure', 'attribute', 'Completeness',
            'check_name', 'Regions with no evaluated flood exposure',
            'failed_rows', CAST(m.q5_no_exposure_regions AS BIGINT), 'total_rows', CAST(m.q5_compared_regions AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Columns: exposure_area_status. NOT_EVALUATED means no safe boundary. Regions: ', COALESCE(NULLIF(m.q5_no_exposure_detail, ''), 'none'))
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_flood_control_exposure', 'attribute', 'Auditability',
            'check_name', 'DPWH and MGB snapshot IDs are the current snapshots',
            'failed_rows', CAST(m.q5_lineage_breaks AS BIGINT), 'total_rows', CAST(m.q5_output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: source_snapshot_id, mgb_source_snapshot_id (D-29)'
        )
    ))
    FROM analytics_validation_q5_metrics AS m
),

q6_checks AS (
    SELECT INLINE(ARRAY(
        NAMED_STRUCT(
            'table_name', 'analytics_region_category', 'attribute', 'Uniqueness',
            'check_name', 'One row per region and sector',
            'failed_rows', CAST(m.q6_output_rows - m.q6_output_keys AS BIGINT), 'total_rows', CAST(m.q6_output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_key, standardized_sector'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_region_category', 'attribute', 'Completeness',
            'check_name', 'Every Gold region and sector pair has exactly one row',
            'failed_rows', CAST(m.q6_group_set_breaks AS BIGINT), 'total_rows', CAST(m.q6_compared_groups AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_key, standardized_sector'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_region_category', 'attribute', 'Accuracy',
            'check_name', 'Project count, projects with a budget, and budget match Gold for every region and sector',
            'failed_rows', CAST(m.q6_count_budget_breaks AS BIGINT), 'total_rows', CAST(m.q6_compared_groups AS BIGINT),
            'action', 'stop',
            'details', 'Columns: project_count, projects_with_budget, total_reported_budget_pesos. Compared with a NULL-safe equal'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_region_category', 'attribute', 'Accuracy',
            'check_name', 'Average budget divides by projects with a budget',
            'failed_rows', CAST(m.q6_average_breaks AS BIGINT), 'total_rows', CAST(m.q6_compared_groups AS BIGINT),
            'action', 'stop',
            'details', 'Columns: avg_budget_per_project_pesos, projects_with_budget'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_region_category', 'attribute', 'Accuracy',
            'check_name', 'Share of region budget and share of sector budget match Gold',
            'failed_rows', CAST(m.q6_share_breaks AS BIGINT), 'total_rows', CAST(m.q6_compared_groups AS BIGINT),
            'action', 'stop',
            'details', 'Columns: share_of_region_budget_pct, share_of_sector_budget_pct'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_region_category', 'attribute', 'Consistency',
            'check_name', 'Each region adds up to its Q1 projects, budget, and coverage status',
            'failed_rows', CAST(m.q6_region_breaks AS BIGINT), 'total_rows', CAST(m.q6_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_key, project_count, total_reported_budget_pesos, source_coverage_status'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_region_category', 'attribute', 'Completeness',
            'check_name', 'A region with no mapped project keeps exactly one empty row',
            'failed_rows', CAST(m.q6_empty_row_breaks AS BIGINT), 'total_rows', CAST(m.q6_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: standardized_sector, project_count'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_region_category', 'attribute', 'Auditability',
            'check_name', 'Snapshot ID is the current DPWH snapshot',
            'failed_rows', CAST(m.q6_lineage_breaks AS BIGINT), 'total_rows', CAST(m.q6_output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: source_snapshot_id (D-29)'
        )
    ))
    FROM analytics_validation_q6_metrics AS m
),

year_trend_checks AS (
    SELECT INLINE(ARRAY(
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_year_trend', 'attribute', 'Uniqueness',
            'check_name', 'One row per infrastructure year, including NULL',
            'failed_rows', CAST(m.yt_output_rows - m.yt_output_keys AS BIGINT), 'total_rows', CAST(m.yt_output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: infra_year'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_year_trend', 'attribute', 'Completeness',
            'check_name', 'Every Gold infrastructure year, including NULL, has exactly one row',
            'failed_rows', CAST(m.yt_row_set_breaks AS BIGINT), 'total_rows', CAST(m.yt_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: infra_year. Compared with a NULL-safe equal'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_year_trend', 'attribute', 'Consistency',
            'check_name', 'Every current project finds its dim_project and dim_project_status row',
            'failed_rows', CAST(m.yt_unjoined_projects AS BIGINT), 'total_rows', CAST(m.yt_gold_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: fact_project_snapshot.project_key, fact_project_snapshot.status_key'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_year_trend', 'attribute', 'Accuracy',
            'check_name', 'Project count, projects with a budget, and budget match Gold for every year and a NULL budget stays NULL',
            'failed_rows', CAST(m.yt_count_budget_breaks AS BIGINT), 'total_rows', CAST(m.yt_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: project_count, projects_with_budget, total_reported_budget_pesos. Compared with a NULL-safe equal'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_year_trend', 'attribute', 'Accuracy',
            'check_name', 'Budget coverage and average budget divide by projects with a budget',
            'failed_rows', CAST(m.yt_average_breaks AS BIGINT), 'total_rows', CAST(m.yt_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: budget_coverage_pct, avg_budget_per_project_pesos, projects_with_budget'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_year_trend', 'attribute', 'Accuracy',
            'check_name', 'Status group counts match Gold',
            'failed_rows', CAST(m.yt_status_breaks AS BIGINT), 'total_rows', CAST(m.yt_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: finished_count, active_count, inactive_count, unmapped_status_count, dim_project_status.status_group'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_year_trend', 'attribute', 'Consistency',
            'check_name', 'The four status counts add up to the project count',
            'failed_rows', CAST(m.yt_status_sum_breaks AS BIGINT), 'total_rows', CAST(m.yt_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: finished_count, active_count, inactive_count, unmapped_status_count, project_count'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_year_trend', 'attribute', 'Accuracy',
            'check_name', 'For Procurement count matches Gold',
            'failed_rows', CAST(m.yt_for_procurement_breaks AS BIGINT), 'total_rows', CAST(m.yt_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: for_procurement_count, dim_project_status.source_status'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_year_trend', 'attribute', 'Accuracy',
            'check_name', 'Long-running count and its evaluated denominator match Gold',
            'failed_rows', CAST(m.yt_long_running_breaks AS BIGINT), 'total_rows', CAST(m.yt_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: long_running_evaluated_count, long_running_count. The denominator counts projects whose flag is not NULL'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_year_trend', 'attribute', 'Accuracy',
            'check_name', 'Zero-progress count and its evaluated denominator match Gold',
            'failed_rows', CAST(m.yt_zero_progress_breaks AS BIGINT), 'total_rows', CAST(m.yt_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: zero_progress_evaluated_count, zero_progress_count. The denominator counts projects whose flag is not NULL'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_year_trend', 'attribute', 'Accuracy',
            'check_name', 'Long-running and zero-progress rates divide by projects where the flag is known',
            'failed_rows', CAST(m.yt_rate_breaks AS BIGINT), 'total_rows', CAST(m.yt_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: pct_long_running, pct_zero_progress'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_year_trend', 'attribute', 'Validity',
            'check_name', 'Flag counts never exceed their denominators',
            'failed_rows', CAST(m.yt_impossible_rows AS BIGINT), 'total_rows', CAST(m.yt_output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: project_count, long_running_evaluated_count, long_running_count, zero_progress_evaluated_count, zero_progress_count'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_year_trend', 'attribute', 'Consistency',
            'check_name', 'Years, including NULL, add up to every current Gold project and budget',
            'failed_rows', CAST(CASE WHEN m.yt_output_projects <> m.yt_gold_projects OR m.yt_output_budget <> m.yt_gold_budget THEN 1 ELSE 0 END AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: project_count, total_reported_budget_pesos. View projects=', m.yt_output_projects, ', Gold=', m.yt_gold_projects, ', of which NULL year=', m.yt_null_year_projects, '. View budget=', m.yt_output_budget, ', Gold=', m.yt_gold_budget)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_year_trend', 'attribute', 'Auditability',
            'check_name', 'Snapshot ID is the current DPWH snapshot',
            'failed_rows', CAST(m.yt_lineage_breaks AS BIGINT), 'total_rows', CAST(m.yt_output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: source_snapshot_id (D-29)'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_year_trend', 'attribute', 'Completeness',
            'check_name', 'Projects with no infrastructure year (infra_year NULL)',
            'failed_rows', CAST(m.yt_null_year_projects AS BIGINT), 'total_rows', CAST(m.yt_gold_rows AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Columns: infra_year. They keep their own NULL row. Reported budget=', m.yt_null_year_budget, '. Observed years ', COALESCE(CAST(m.yt_min_year AS STRING), 'none'), ' to ', COALESCE(CAST(m.yt_max_year AS STRING), 'none'), '. Projects by year: ', COALESCE(m.yt_year_detail, 'none'))
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_year_trend', 'attribute', 'Completeness',
            'check_name', 'Projects without a long-running flag',
            'failed_rows', CAST(m.yt_unevaluated_long_running AS BIGINT), 'total_rows', CAST(m.yt_gold_rows AS BIGINT),
            'action', 'flag',
            'details', 'Columns: long_running_flag, long_running_evaluated_count. They are left out of pct_long_running'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_year_trend', 'attribute', 'Consistency',
            'check_name', 'Zero-progress projects that are For Procurement',
            'failed_rows', CAST(m.yt_zero_progress_for_procurement AS BIGINT), 'total_rows', CAST(m.yt_zero_progress_projects AS BIGINT),
            'action', 'flag',
            'details', 'Columns: zero_progress_flag, dim_project_status.source_status. pct_zero_progress includes them (D-46). They are not stalled'
        )
    ))
    FROM analytics_validation_year_trend_metrics AS m
),

year_sector_checks AS (
    SELECT INLINE(ARRAY(
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_year_sector', 'attribute', 'Uniqueness',
            'check_name', 'One row per infrastructure year, sector, and mapping status',
            'failed_rows', CAST(m.ys_output_rows - m.ys_output_keys AS BIGINT), 'total_rows', CAST(m.ys_output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: infra_year, standardized_sector, infra_type_mapping_status'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_year_sector', 'attribute', 'Completeness',
            'check_name', 'Every Gold year and sector combination has exactly one row',
            'failed_rows', CAST(m.ys_row_set_breaks AS BIGINT), 'total_rows', CAST(m.ys_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: infra_year, standardized_sector, infra_type_mapping_status. Compared with a NULL-safe equal'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_year_sector', 'attribute', 'Accuracy',
            'check_name', 'Project count, projects with a budget, and budget match Gold for every year and sector',
            'failed_rows', CAST(m.ys_count_budget_breaks AS BIGINT), 'total_rows', CAST(m.ys_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: project_count, projects_with_budget, total_reported_budget_pesos. Compared with a NULL-safe equal'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_year_sector', 'attribute', 'Accuracy',
            'check_name', 'Average budget divides by projects with a budget',
            'failed_rows', CAST(m.ys_average_breaks AS BIGINT), 'total_rows', CAST(m.ys_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: avg_budget_per_project_pesos, projects_with_budget'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_year_sector', 'attribute', 'Accuracy',
            'check_name', 'Share of the year budget matches Gold',
            'failed_rows', CAST(m.ys_share_breaks AS BIGINT), 'total_rows', CAST(m.ys_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: share_of_year_budget_pct. The denominator is the reported budget of the same infra_year'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_year_sector', 'attribute', 'Accuracy',
            'check_name', 'The sectors of each year add up to 100 percent',
            'failed_rows', CAST(m.ys_share_total_breaks AS BIGINT), 'total_rows', CAST(m.ys_years AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: share_of_year_budget_pct. Tolerance=', share_total_tolerance_pct)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_year_sector', 'attribute', 'Consistency',
            'check_name', 'Each year adds up to its trend-view projects and budget',
            'failed_rows', CAST(m.ys_trend_breaks AS BIGINT), 'total_rows', CAST(m.ys_years AS BIGINT),
            'action', 'stop',
            'details', 'Columns: project_count, total_reported_budget_pesos, analytics_infrastructure_year_trend'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_year_sector', 'attribute', 'Consistency',
            'check_name', 'Years and sectors add up to every current Gold project and budget',
            'failed_rows', CAST(CASE WHEN m.ys_output_projects <> m.ys_gold_projects OR m.ys_output_budget <> m.ys_gold_budget THEN 1 ELSE 0 END AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: project_count, total_reported_budget_pesos. View projects=', m.ys_output_projects, ', Gold=', m.ys_gold_projects, '. View budget=', m.ys_output_budget, ', Gold=', m.ys_gold_budget)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_infrastructure_year_sector', 'attribute', 'Auditability',
            'check_name', 'Snapshot ID is the current DPWH snapshot',
            'failed_rows', CAST(m.ys_lineage_breaks AS BIGINT), 'total_rows', CAST(m.ys_output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: source_snapshot_id (D-29)'
        )
    ))
    FROM analytics_validation_year_sector_metrics AS m
),

delivery_cohort_checks AS (
    SELECT INLINE(ARRAY(
        NAMED_STRUCT(
            'table_name', 'analytics_delivery_cohort', 'attribute', 'Uniqueness',
            'check_name', 'One row per infrastructure year, status group, and status mapping status',
            'failed_rows', CAST(m.dc_output_rows - m.dc_output_keys AS BIGINT), 'total_rows', CAST(m.dc_output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: infra_year, status_group, status_mapping_status'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_delivery_cohort', 'attribute', 'Completeness',
            'check_name', 'Every Gold year and status group has exactly one row',
            'failed_rows', CAST(m.dc_row_set_breaks AS BIGINT), 'total_rows', CAST(m.dc_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: infra_year, status_group, status_mapping_status. Compared with a NULL-safe equal'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_delivery_cohort', 'attribute', 'Accuracy',
            'check_name', 'Project count and For Procurement count match Gold for every row',
            'failed_rows', CAST(m.dc_count_breaks AS BIGINT), 'total_rows', CAST(m.dc_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: project_count, for_procurement_count'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_delivery_cohort', 'attribute', 'Accuracy',
            'check_name', 'Long-running count and its evaluated denominator match Gold',
            'failed_rows', CAST(m.dc_long_running_breaks AS BIGINT), 'total_rows', CAST(m.dc_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: long_running_evaluated_count, long_running_count. The denominator counts projects whose flag is not NULL'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_delivery_cohort', 'attribute', 'Accuracy',
            'check_name', 'Zero-progress count and its evaluated denominator match Gold',
            'failed_rows', CAST(m.dc_zero_progress_breaks AS BIGINT), 'total_rows', CAST(m.dc_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: zero_progress_evaluated_count, zero_progress_count. The denominator counts projects whose flag is not NULL'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_delivery_cohort', 'attribute', 'Accuracy',
            'check_name', 'Long-running and zero-progress rates divide by projects where the flag is known',
            'failed_rows', CAST(m.dc_rate_breaks AS BIGINT), 'total_rows', CAST(m.dc_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: pct_long_running, pct_zero_progress'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_delivery_cohort', 'attribute', 'Validity',
            'check_name', 'Flag counts never exceed their denominators',
            'failed_rows', CAST(m.dc_impossible_rows AS BIGINT), 'total_rows', CAST(m.dc_output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: project_count, long_running_evaluated_count, long_running_count, zero_progress_evaluated_count, zero_progress_count'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_delivery_cohort', 'attribute', 'Validity',
            'check_name', 'Status mapping status follows the status group',
            'failed_rows', CAST(m.dc_reason_breaks AS BIGINT), 'total_rows', CAST(m.dc_output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: status_group, status_mapping_status. MAPPED with a group, NO_SOURCE_STATUS or NO_APPROVED_STATUS_RULE without one'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_delivery_cohort', 'attribute', 'Accuracy',
            'check_name', 'Share of year projects divides by every project of the same year',
            'failed_rows', CAST(m.dc_year_share_breaks AS BIGINT), 'total_rows', CAST(m.dc_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: pct_of_year_projects. The denominator is counted from Gold and includes projects with no status group'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_delivery_cohort', 'attribute', 'Accuracy',
            'check_name', 'The rows of each year add up to 100 percent',
            'failed_rows', CAST(m.dc_share_total_breaks AS BIGINT), 'total_rows', CAST(m.dc_years AS BIGINT),
            'action', 'stop',
            'details', 'Columns: pct_of_year_projects. Each row is rounded to 2 decimals, so the tolerance is 0.005 per row of the year'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_delivery_cohort', 'attribute', 'Consistency',
            'check_name', 'Each year adds up to its trend-view projects, flag denominators, and For Procurement count',
            'failed_rows', CAST(m.dc_trend_breaks AS BIGINT), 'total_rows', CAST(m.dc_years AS BIGINT),
            'action', 'stop',
            'details', 'Columns: project_count, long_running_evaluated_count, zero_progress_evaluated_count, for_procurement_count, analytics_infrastructure_year_trend'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_delivery_cohort', 'attribute', 'Auditability',
            'check_name', 'Snapshot ID is the current DPWH snapshot',
            'failed_rows', CAST(m.dc_lineage_breaks AS BIGINT), 'total_rows', CAST(m.dc_output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: source_snapshot_id (D-29)'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_delivery_cohort', 'attribute', 'Completeness',
            'check_name', 'Projects with no mapped status group (status_group NULL)',
            'failed_rows', CAST(m.dc_unmapped_projects AS BIGINT), 'total_rows', CAST(m.dc_expected_projects AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Columns: status_group, status_mapping_status. NO_SOURCE_STATUS=', m.dc_no_source_status_projects, ', NO_APPROVED_STATUS_RULE=', m.dc_no_rule_projects, '. They keep their own rows')
        )
    ))
    FROM analytics_validation_delivery_cohort_metrics AS m
),

size_checks AS (
    SELECT INLINE(ARRAY(
        NAMED_STRUCT(
            'table_name', 'analytics_project_size_concentration', 'attribute', 'Uniqueness',
            'check_name', 'One row per region',
            'failed_rows', CAST(m.ps_output_rows - m.ps_output_regions AS BIGINT), 'total_rows', CAST(m.ps_output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_key'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_project_size_concentration', 'attribute', 'Completeness',
            'check_name', 'Every Q1 region has exactly one row',
            'failed_rows', CAST(m.ps_region_set_breaks AS BIGINT), 'total_rows', CAST(m.ps_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_key'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_project_size_concentration', 'attribute', 'Validity',
            'check_name', 'Region key 0 is never in the view',
            'failed_rows', CAST(m.ps_key0_rows AS BIGINT), 'total_rows', CAST(m.ps_output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_key. Q1 counts key 0 projects separately'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_project_size_concentration', 'attribute', 'Consistency',
            'check_name', 'Coverage status, project counts, budget, and average match Q1',
            'failed_rows', CAST(m.ps_q1_breaks AS BIGINT), 'total_rows', CAST(m.ps_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: source_coverage_status, project_count, projects_with_budget, budget_coverage_pct, total_reported_budget_pesos, avg_budget_per_project_pesos'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_project_size_concentration', 'attribute', 'Accuracy',
            'check_name', 'Project count, projects with a budget, and budget match Gold for every region',
            'failed_rows', CAST(m.ps_gold_breaks AS BIGINT), 'total_rows', CAST(m.ps_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: project_count, projects_with_budget, total_reported_budget_pesos. Compared with a NULL-safe equal'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_project_size_concentration', 'attribute', 'Accuracy',
            'check_name', 'Projects with a budget of 0 match Gold and a NULL budget is not counted',
            'failed_rows', CAST(m.ps_zero_budget_breaks AS BIGINT), 'total_rows', CAST(m.ps_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: projects_with_zero_budget. Gold rows where reported_budget_pesos = 0'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_project_size_concentration', 'attribute', 'Accuracy',
            'check_name', 'Median, 90th percentile, and largest budget match Gold over projects with a known budget',
            'failed_rows', CAST(m.ps_distribution_breaks AS BIGINT), 'total_rows', CAST(m.ps_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: median_budget_per_project_pesos, p90_budget_per_project_pesos, max_budget_per_project_pesos. Rebuilt from Gold rows where reported_budget_pesos IS NOT NULL with the same exact PERCENTILE_CONT'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_project_size_concentration', 'attribute', 'Accuracy',
            'check_name', 'Largest project share divides the largest budget by the region budget',
            'failed_rows', CAST(m.ps_largest_share_breaks AS BIGINT), 'total_rows', CAST(m.ps_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: largest_project_share_of_region_budget_pct'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_project_size_concentration', 'attribute', 'Validity',
            'check_name', 'The median is at most the 90th percentile, which is at most the largest budget',
            'failed_rows', CAST(m.ps_order_breaks AS BIGINT), 'total_rows', CAST(m.ps_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: median_budget_per_project_pesos, p90_budget_per_project_pesos, max_budget_per_project_pesos'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_project_size_concentration', 'attribute', 'Validity',
            'check_name', 'A region with no mapped project shows no distribution value',
            'failed_rows', CAST(m.ps_no_project_value_breaks AS BIGINT), 'total_rows', CAST(m.ps_compared_regions AS BIGINT),
            'action', 'stop',
            'details', 'Columns: project_count, median_budget_per_project_pesos, p90_budget_per_project_pesos, max_budget_per_project_pesos, largest_project_share_of_region_budget_pct'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_project_size_concentration', 'attribute', 'Auditability',
            'check_name', 'Snapshot ID is the current DPWH snapshot',
            'failed_rows', CAST(m.ps_lineage_breaks AS BIGINT), 'total_rows', CAST(m.ps_output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: source_snapshot_id (D-29)'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_project_size_concentration', 'attribute', 'Completeness',
            'check_name', 'Projects in the view with a reported budget of 0',
            'failed_rows', CAST(m.ps_zero_budget_projects AS BIGINT), 'total_rows', CAST(m.ps_projects AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Columns: projects_with_zero_budget. A 0 stays 0 as reported and stays in the median and p90. For Procurement projects report 0 (D-46). Projects with a known budget=', m.ps_projects_with_budget)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_project_size_concentration', 'attribute', 'Completeness',
            'check_name', 'Official regions with no or limited DPWH project coverage',
            'failed_rows', CAST(m.ps_unreported_regions AS BIGINT), 'total_rows', CAST(m.ps_compared_regions AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Columns: source_coverage_status. Not normal regional coverage: ', COALESCE(NULLIF(m.ps_unreported_detail, ''), 'none'))
        )
    ))
    FROM analytics_validation_size_metrics AS m
),

office_checks AS (
    SELECT INLINE(ARRAY(
        NAMED_STRUCT(
            'table_name', 'analytics_implementing_office_portfolio', 'attribute', 'Uniqueness',
            'check_name', 'One row per implementing office, including NULL',
            'failed_rows', CAST(m.io_output_rows - m.io_output_keys AS BIGINT), 'total_rows', CAST(m.io_output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: implementing_office'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_implementing_office_portfolio', 'attribute', 'Completeness',
            'check_name', 'Every Gold office, including NULL, has exactly one row',
            'failed_rows', CAST(m.io_row_set_breaks AS BIGINT), 'total_rows', CAST(m.io_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: implementing_office. Compared with a NULL-safe equal'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_implementing_office_portfolio', 'attribute', 'Accuracy',
            'check_name', 'Project count, projects with a budget, and budget match Gold for every office and a NULL budget stays NULL',
            'failed_rows', CAST(m.io_count_budget_breaks AS BIGINT), 'total_rows', CAST(m.io_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: project_count, projects_with_budget, total_reported_budget_pesos. Compared with a NULL-safe equal'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_implementing_office_portfolio', 'attribute', 'Accuracy',
            'check_name', 'Average and median budget use only projects with a known budget',
            'failed_rows', CAST(m.io_average_breaks AS BIGINT), 'total_rows', CAST(m.io_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: avg_budget_per_project_pesos, median_budget_per_project_pesos, projects_with_budget'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_implementing_office_portfolio', 'attribute', 'Accuracy',
            'check_name', 'Region and sector counts match Gold',
            'failed_rows', CAST(m.io_spread_breaks AS BIGINT), 'total_rows', CAST(m.io_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_count, sector_count. Region key 0 and NULL sectors are not counted'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_implementing_office_portfolio', 'attribute', 'Accuracy',
            'check_name', 'Long-running and zero-progress counts and their evaluated denominators match Gold',
            'failed_rows', CAST(m.io_flag_count_breaks AS BIGINT), 'total_rows', CAST(m.io_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: long_running_evaluated_count, long_running_count, zero_progress_evaluated_count, zero_progress_count'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_implementing_office_portfolio', 'attribute', 'Accuracy',
            'check_name', 'Long-running and zero-progress rates divide by projects where the flag is known',
            'failed_rows', CAST(m.io_rate_breaks AS BIGINT), 'total_rows', CAST(m.io_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: pct_long_running, pct_zero_progress'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_implementing_office_portfolio', 'attribute', 'Validity',
            'check_name', 'Flag counts never exceed their denominators',
            'failed_rows', CAST(m.io_impossible_rows AS BIGINT), 'total_rows', CAST(m.io_output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: project_count, long_running_evaluated_count, long_running_count, zero_progress_evaluated_count, zero_progress_count'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_implementing_office_portfolio', 'attribute', 'Accuracy',
            'check_name', 'Ranks cover only real offices, ties share a rank, and rank 1 is the highest',
            'failed_rows', CAST(m.io_rank_breaks AS BIGINT), 'total_rows', CAST(m.io_compared_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: project_count_rank, budget_rank'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_implementing_office_portfolio', 'attribute', 'Validity',
            'check_name', 'The NULL office row has no rank',
            'failed_rows', CAST(m.io_null_office_rank_breaks AS BIGINT), 'total_rows', CAST(m.io_output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: implementing_office, project_count_rank, budget_rank'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_implementing_office_portfolio', 'attribute', 'Consistency',
            'check_name', 'Offices, including NULL, add up to every current Gold project and budget',
            'failed_rows', CAST(CASE WHEN m.io_output_projects <> m.io_gold_projects OR m.io_output_budget <> m.io_gold_budget THEN 1 ELSE 0 END AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: project_count, total_reported_budget_pesos. View projects=', m.io_output_projects, ', Gold=', m.io_gold_projects, ', of which NULL office=', m.io_null_office_projects, '. View budget=', m.io_output_budget, ', Gold=', m.io_gold_budget)
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_implementing_office_portfolio', 'attribute', 'Auditability',
            'check_name', 'Snapshot ID is the current DPWH snapshot',
            'failed_rows', CAST(m.io_lineage_breaks AS BIGINT), 'total_rows', CAST(m.io_output_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: source_snapshot_id (D-29)'
        ),
        NAMED_STRUCT(
            'table_name', 'analytics_implementing_office_portfolio', 'attribute', 'Completeness',
            'check_name', 'Projects with no resolved implementing office (implementing_office NULL)',
            'failed_rows', CAST(m.io_null_office_projects AS BIGINT), 'total_rows', CAST(m.io_gold_rows AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Columns: implementing_office. They keep their own NULL row with no rank. Gold does not say whether the office was missing or conflicting. Reported budget=', m.io_null_office_budget, '. Real offices=', m.io_real_offices)
        )
    ))
    FROM analytics_validation_office_metrics AS m
),

checks AS (
    -- each metrics row is turned into check rows, so no table is scanned again per check
    SELECT * FROM q1_checks
    UNION ALL
    SELECT * FROM q2_checks
    UNION ALL
    SELECT * FROM q3_checks
    UNION ALL
    SELECT * FROM q4_checks
    UNION ALL
    SELECT * FROM q5_checks
    UNION ALL
    SELECT * FROM q6_checks
    UNION ALL
    SELECT * FROM year_trend_checks
    UNION ALL
    SELECT * FROM year_sector_checks
    UNION ALL
    SELECT * FROM delivery_cohort_checks
    UNION ALL
    SELECT * FROM size_checks
    UNION ALL
    SELECT * FROM office_checks
)

SELECT
    context.validation_run_id,
    COALESCE(
        CASE checks.table_name
            WHEN 'analytics_population_equity' THEN context.population_fact_run_id
            WHEN 'analytics_flood_control_exposure' THEN context.exposure_fact_run_id
            ELSE context.project_fact_run_id
        END,
        context.validation_run_id
    ) AS pipeline_run_id,
    'analytics' AS layer,
    checks.table_name,
    CASE checks.table_name
        WHEN 'analytics_flood_control_exposure' THEN context.mgb_source_snapshot_id
        ELSE context.project_snapshot_id
    END AS source_snapshot_id,
    validation_rule_version AS rule_version,
    checks.attribute AS data_quality_attribute,
    checks.check_name AS data_quality_check,
    checks.failed_rows,
    checks.total_rows,
    CASE WHEN checks.total_rows > 0
        THEN ROUND(100.0 * checks.failed_rows / checks.total_rows, 4) END AS percentage,
    CASE
        WHEN checks.failed_rows = 0 THEN 'PASS'
        WHEN checks.action = 'stop' THEN 'FAIL'
        ELSE 'FLAG'
    END AS status,
    checks.action,
    checks.details,
    CURRENT_TIMESTAMP() AS run_ts
FROM checks
CROSS JOIN analytics_validation_context AS context;

SELECT
    table_name,
    data_quality_attribute,
    data_quality_check,
    status,
    failed_rows,
    total_rows
FROM analytics_validation_checks
ORDER BY CASE status WHEN 'FAIL' THEN 1 WHEN 'FLAG' THEN 2 ELSE 3 END, table_name, data_quality_check;

## 15. Save the evidence before the gate

**What:** `MERGE` the checks into `04-validation.analytics_dq_results`, then show them, failures first.

**Why:** the saved rows stay reviewable even when the gate stops the run.

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS `04-validation`.analytics_dq_results (
    validation_run_id STRING NOT NULL, pipeline_run_id STRING NOT NULL, layer STRING NOT NULL,
    table_name STRING NOT NULL, source_snapshot_id STRING, rule_version STRING,
    data_quality_attribute STRING NOT NULL, data_quality_check STRING NOT NULL,
    failed_rows BIGINT, total_rows BIGINT, percentage DOUBLE, status STRING NOT NULL,
    action STRING NOT NULL, details STRING, run_ts TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Analytics validation evidence. One row per check per validation run.';

MERGE INTO `04-validation`.analytics_dq_results AS target
USING analytics_validation_checks AS source
ON target.validation_run_id = source.validation_run_id
AND target.table_name = source.table_name
AND target.data_quality_check = source.data_quality_check
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

SELECT
    table_name,
    status,
    action,
    failed_rows,
    total_rows,
    percentage,
    data_quality_check,
    details
FROM `04-validation`.analytics_dq_results
WHERE validation_run_id = (SELECT validation_run_id FROM analytics_validation_context)
ORDER BY CASE status WHEN 'FAIL' THEN 1 WHEN 'FLAG' THEN 2 ELSE 3 END, table_name, data_quality_check;

## 16. STOP gate

The run stops here if any STOP check failed.

In [ ]:
%sql
SELECT ASSERT_TRUE(
    COUNT(*) > 0 AND COUNT_IF(status = 'FAIL' AND action = 'stop') = 0,
    CONCAT(
        COUNT_IF(status = 'FAIL' AND action = 'stop'),
        ' blocking analytics checks failed or no evidence was saved'
    )
)
FROM `04-validation`.analytics_dq_results
WHERE validation_run_id = (SELECT validation_run_id FROM analytics_validation_context);

## Summary

This notebook checks all eleven analytics views against Gold for grain, row sets, every measure, ranks, totals, lineage, and the coverage and exposure rules: the six official business-question views and the five exploratory monitoring views (D-54, proposed). Every view carries the snapshot it was built from. Known data limits stay visible as FLAG rows with counts: projects on key `0`, missing and zero budgets, unmapped categories, Multi-sector flood projects, unknown statuses, missing start dates, regions left out of equity ratios, regions with overlapping or missing flood exposure, projects without an `infra_year` or a resolved implementing office, and zero-progress projects that are For Procurement. The rule version is `analytics-validation-v2`. Run evidence for the six official views under v1 is in [docs/evidence/analytics-views](../../docs/evidence/analytics-views/README.md). Run evidence for all eleven views under v2 is in [docs/evidence/exploratory-analytics-views](../../docs/evidence/exploratory-analytics-views/README.md).